# Regression Modeling — Stage 1 and Stage 2

## 0. Project Objective

This notebook builds a safe base for later regression models. It checks two processed loan datasets, creates one shared split, creates shared cross-validation folds, defines metrics, and saves audit artifacts. It does not train a real model.

## 1. Imports and Configuration

One configuration keeps paths and fixed settings in one place. Fixed seeds make the split and folds repeatable.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split

PROJECT_ROOT = Path.cwd().resolve()
required_root_markers = [PROJECT_ROOT / "data" / "regression_with_sensitive_features.csv", PROJECT_ROOT / "AGENTS.md"]
if not all(path.exists() for path in required_root_markers):
    raise RuntimeError("Run this notebook from the regresionpart2 project root. Required root markers were not found.")
CONFIG = {
    "project_root": PROJECT_ROOT,
    "with_sensitive_path": PROJECT_ROOT / "data" / "regression_with_sensitive_features.csv",
    "without_sensitive_path": PROJECT_ROOT / "data" / "regression_without_sensitive_features.csv",
    "source_notebook_path": (PROJECT_ROOT / ".." / "main" / "REGRESION_PART1.ipynb").resolve(),
    "artifact_root": PROJECT_ROOT / "artifacts",
    "target_column": "loan_amount_000s",
    "target_unit": "thousands of US dollars",
    "random_state": 42,
    "test_size": 0.20,
    "n_cv_folds": 3,
    "n_target_bins": 10,
    "encoding": "utf-8-sig",
    "display_rows": 8,
}
ARTIFACT_DIRS = {
    "backups": CONFIG["artifact_root"] / "backups",
    "data_contract": CONFIG["artifact_root"] / "data_contract",
    "splits": CONFIG["artifact_root"] / "splits",
    "results": CONFIG["artifact_root"] / "results",
    "reports": CONFIG["artifact_root"] / "reports",
}
for directory in ARTIFACT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])
pd.set_option("display.max_rows", CONFIG["display_rows"])
pd.set_option("display.max_columns", 12)
print({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()})

# Later stages use saved Stage 1 artifacts without recreating them.
CURRENT_PROJECT_STAGE = 2

{'project_root': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\regresionpart2', 'with_sensitive_path': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\regresionpart2\\data\\regression_with_sensitive_features.csv', 'without_sensitive_path': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\regresionpart2\\data\\regression_without_sensitive_features.csv', 'source_notebook_path': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\main\\REGRESION_PART1.ipynb', 'artifact_root': 'D:\\SHARIF\\TERM7\\DATA\\PROJECT\\regresionpart2\\artifacts', 'target_column': 'loan_amount_000s', 'target_unit': 'thousands of US dollars', 'random_state': 42, 'test_size': 0.2, 'n_cv_folds': 3, 'n_target_bins': 10, 'encoding': 'utf-8-sig', 'display_rows': 8}


The project uses seed 42, a 20 percent test set, three CV folds, and ten requested target bins. The target unit was confirmed in the Part 1 notebook.

## 2. Project and File Discovery

File discovery prevents the notebook from using a guessed input. The exact CSV names are required. The source notebook is the unique close match in the project.

In [2]:
required_paths = {
    "with_sensitive_csv": CONFIG["with_sensitive_path"],
    "without_sensitive_csv": CONFIG["without_sensitive_path"],
    "source_notebook": CONFIG["source_notebook_path"],
}
discovery = []
for name, path in required_paths.items():
    discovery.append({
        "name": name,
        "relative_or_resolved_path": str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path),
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
    })
discovery_df = pd.DataFrame(discovery)
display(discovery_df)
if not discovery_df["exists"].all():
    raise FileNotFoundError("A required source file was not found.")

,name,relative_or_resolved_path,exists,size_bytes
0,with_sensitive_csv,data\regression_with_sensitive_features.csv,True,357758268
1,without_sensitive_csv,data\regression_without_sensitive_features.csv,True,278054553
2,source_notebook,D:\SHARIF\TERM7\DATA\PROJECT\main\REGRESION_PA...,True,4196401


Both processed CSV files and the Part 1 notebook were found. Derived files will be saved only under `artifacts/`.

## 3. Source Data Protection

Hashes, sizes, and modification times provide a source safety record. Hashing reads each file in small blocks and does not load the full file into memory.

In [3]:
def sha256_stream(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def file_fingerprint(path):
    path = Path(path).resolve()
    stat = path.stat()
    return {
        "resolved_path": str(path),
        "size_bytes": stat.st_size,
        "modified_time_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
        "sha256": sha256_stream(path),
    }

source_paths = {
    "with_sensitive": CONFIG["with_sensitive_path"],
    "without_sensitive": CONFIG["without_sensitive_path"],
}
protected_source_paths = {**source_paths, "part1_notebook": CONFIG["source_notebook_path"]}
source_hashes_before = {name: file_fingerprint(path) for name, path in protected_source_paths.items()}
before_path = ARTIFACT_DIRS["data_contract"] / "source_hashes_before.json"
if before_path.exists():
    source_hashes_before = json.loads(before_path.read_text(encoding="utf-8"))
else:
    before_path.write_text(json.dumps(source_hashes_before, indent=2), encoding="utf-8")
display(pd.DataFrame(source_hashes_before).T[["size_bytes", "sha256"]])

,size_bytes,sha256
with_sensitive,357758268,6dc52dca5a8a7196a75213fab4a5a5c0a541f843902194...
without_sensitive,278054553,e90f7bb49cce5584c7ab250c1db6a107de5cf640c7839f...
part1_notebook,4196401,990ced79600bfa4d7a0ba0bf3326b85812f36faa4df036...


Source fingerprints were saved before the full data load. The final section will calculate them again and require exact hash matches.

## 4. Data Loading

The source files are loaded into canonical DataFrames. Text columns use the category dtype to reduce memory use. No source DataFrame is changed in place.

In [4]:
def load_source_csv(path):
    sample = pd.read_csv(path, nrows=5000, encoding=CONFIG["encoding"])
    dtype_map = {col: "category" for col in sample.select_dtypes(include="object").columns}
    return pd.read_csv(path, dtype=dtype_map, encoding=CONFIG["encoding"], low_memory=False)

df_with_sensitive_raw = load_source_csv(CONFIG["with_sensitive_path"])
df_without_sensitive_raw = load_source_csv(CONFIG["without_sensitive_path"])

def compact_frame_summary(name, frame):
    return {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "numeric_columns": frame.select_dtypes(include=np.number).shape[1],
        "categorical_columns": frame.select_dtypes(exclude=np.number).shape[1],
        "memory_mb": round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
    }

loading_summary = pd.DataFrame([
    compact_frame_summary("with_sensitive", df_with_sensitive_raw),
    compact_frame_summary("without_sensitive", df_without_sensitive_raw),
])
display(loading_summary)
display(df_with_sensitive_raw.head(3))

,dataset,rows,columns,numeric_columns,categorical_columns,memory_mb
0,with_sensitive,499736,44,22,22,96.30
1,without_sensitive,499736,36,21,15,89.15


,loan_amount_000s,respondent_id,agency_name,loan_type_name,property_type_name,loan_purpose_name,...,has_co_applicant,loan_program_group,applicant_income_area_group,tract_income_level,majority_minority_tract,us_region
0,603,0000035295,Federal Deposit Insurance Corporation,Conventional,One-to-four family dwelling (other than manufa...,Home purchase,...,1,Conventional,High,High,Not majority minority,South
1,341,33-0941669,Department of Housing and Urban Development,FHA-insured,One-to-four family dwelling (other than manufa...,Refinancing,...,1,Government backed,Moderate,Moderate,Not majority minority,West
2,215,7505400005,Department of Housing and Urban Development,VA-guaranteed,One-to-four family dwelling (other than manufa...,Home purchase,...,0,Government backed,Moderate,Low,Majority minority,West


The sensitive dataset has 44 columns and the non-sensitive dataset has 36 columns. Both contain the same 499,736 observations.

## 5. Data Contract Validation

These checks confirm file, row, common-feature, and target agreement. Comparisons are made one column at a time to limit memory use.

In [5]:
TARGET = CONFIG["target_column"]

def duplicate_column_names(frame):
    return frame.columns[frame.columns.duplicated()].tolist()

def semantically_equal(left, right):
    if not left.isna().equals(right.isna()):
        return False
    if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
        return np.array_equal(left.to_numpy(), right.to_numpy(), equal_nan=True)
    left_text = left.astype("string").fillna("<MISSING>")
    right_text = right.astype("string").fillna("<MISSING>")
    return left_text.equals(right_text)

if duplicate_column_names(df_with_sensitive_raw) or duplicate_column_names(df_without_sensitive_raw):
    raise ValueError("Duplicate column names were found.")
if len(df_with_sensitive_raw) != len(df_without_sensitive_raw):
    raise ValueError("The datasets have different row counts.")
if TARGET not in df_with_sensitive_raw or TARGET not in df_without_sensitive_raw:
    raise KeyError(f"The target {TARGET} is missing.")

common_columns = [c for c in df_without_sensitive_raw.columns if c in df_with_sensitive_raw.columns]
missing_positions_equal = {}
common_values_equal = {}
dtype_differences = []
for column in common_columns:
    left = df_with_sensitive_raw[column]
    right = df_without_sensitive_raw[column]
    missing_positions_equal[column] = left.isna().equals(right.isna())
    common_values_equal[column] = semantically_equal(left, right)
    if str(left.dtype) != str(right.dtype):
        dtype_differences.append({"column": column, "with_dtype": str(left.dtype), "without_dtype": str(right.dtype)})
if not all(missing_positions_equal.values()):
    raise ValueError("Missing-value positions differ in common columns.")
if not all(common_values_equal.values()):
    failed = [c for c, passed in common_values_equal.items() if not passed]
    raise ValueError(f"Common feature values differ: {failed[:10]}")

y = pd.to_numeric(df_with_sensitive_raw[TARGET], errors="raise")
target_checks = {
    "numeric": pd.api.types.is_numeric_dtype(y),
    "no_missing": not y.isna().any(),
    "finite": bool(np.isfinite(y.to_numpy(dtype=float)).all()),
    "positive": bool((y > 0).all()),
    "equal_row_by_row": common_values_equal[TARGET],
}
if not all(target_checks.values()):
    raise ValueError(f"Target validation failed: {target_checks}")

quantiles = y.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
target_summary = {
    "count": int(y.count()), "mean": float(y.mean()), "std": float(y.std()),
    "min": float(y.min()), "p01": float(quantiles.loc[0.01]),
    "p10": float(quantiles.loc[0.10]), "p25": float(quantiles.loc[0.25]),
    "median": float(quantiles.loc[0.50]), "p75": float(quantiles.loc[0.75]),
    "p90": float(quantiles.loc[0.90]), "p95": float(quantiles.loc[0.95]),
    "p99": float(quantiles.loc[0.99]), "max": float(y.max()), "skewness": float(y.skew()),
}
display(pd.DataFrame([target_summary]).T.rename(columns={0: "value"}))
print("Target checks:", target_checks)
print("Common columns checked:", len(common_columns), "| dtype differences:", len(dtype_differences))

,value
count,499736.000000
mean,247.185150
std,302.912039
min,1.000000
...,...
p95,599.000000
p99,1100.000000
max,99000.000000
skewness,113.318462


Target checks: {'numeric': True, 'no_missing': True, 'finite': True, 'positive': True, 'equal_row_by_row': True}
Common columns checked: 36 | dtype differences: 0


All common values and missing positions match by row. The target is positive, finite, and complete. Its skew is very large, so robust metrics and target-bin stratification are useful. No outlier is removed here.

## 6. Target and Feature Review

The feature inventory records types, missing values, cardinality, identifier risks, and leakage warnings without changing the source data.

In [6]:
with_columns = list(df_with_sensitive_raw.columns)
without_columns = list(df_without_sensitive_raw.columns)
sensitive_features = [c for c in with_columns if c not in without_columns]
if TARGET in sensitive_features or not sensitive_features:
    raise ValueError("The sensitive feature difference is invalid.")
if any(c not in with_columns for c in without_columns):
    raise ValueError("A non-sensitive feature is missing from the sensitive dataset.")

def infer_feature_type(column, series):
    lower = column.lower()
    code_like = any(token in lower for token in ["_id", "_code", "tract_number"])
    if code_like or not pd.api.types.is_numeric_dtype(series):
        return "categorical"
    return "numeric"

def cardinality_class(unique_count):
    if unique_count <= 2: return "Binary"
    if unique_count <= 20: return "Low cardinality"
    if unique_count <= 100: return "Medium cardinality"
    return "High cardinality"

inventory_rows = []
for column in with_columns:
    series = df_with_sensitive_raw[column]
    unique_count = int(series.nunique(dropna=True))
    lower = column.lower()
    possible_identifier = any(token in lower for token in ["_id", "_code", "tract_number", "msamd_name", "county_name"])
    possible_leakage = column != TARGET and (TARGET.lower() in lower or "loan_amount" in lower)
    inventory_rows.append({
        "column_name": column,
        "dataset_membership": "both" if column in without_columns else "with_sensitive_only",
        "is_sensitive": column in sensitive_features,
        "pandas_dtype": str(series.dtype),
        "inferred_feature_type": "target" if column == TARGET else infer_feature_type(column, series),
        "missing_count": int(series.isna().sum()),
        "missing_percentage": float(series.isna().mean() * 100),
        "unique_values": unique_count,
        "cardinality_class": cardinality_class(unique_count),
        "is_constant": unique_count <= 1,
        "possible_identifier": possible_identifier,
        "possible_leakage": possible_leakage,
        "is_target": column == TARGET,
    })
feature_inventory = pd.DataFrame(inventory_rows)
feature_inventory.to_csv(ARTIFACT_DIRS["data_contract"] / "feature_inventory.csv", index=False)

def features_of_type(frame, columns, kind):
    return [c for c in columns if c != TARGET and infer_feature_type(c, frame[c]) == kind]

features_with_sensitive = [c for c in with_columns if c != TARGET]
features_without_sensitive = [c for c in without_columns if c != TARGET]
feature_sets = {
    "target_column": TARGET,
    "features_with_sensitive": features_with_sensitive,
    "features_without_sensitive": features_without_sensitive,
    "sensitive_features": sensitive_features,
    "common_features": [c for c in common_columns if c != TARGET],
    "numeric_features_with_sensitive": features_of_type(df_with_sensitive_raw, features_with_sensitive, "numeric"),
    "categorical_features_with_sensitive": features_of_type(df_with_sensitive_raw, features_with_sensitive, "categorical"),
    "numeric_features_without_sensitive": features_of_type(df_without_sensitive_raw, features_without_sensitive, "numeric"),
    "categorical_features_without_sensitive": features_of_type(df_without_sensitive_raw, features_without_sensitive, "categorical"),
    "high_cardinality_features": feature_inventory.loc[(feature_inventory.cardinality_class == "High cardinality") & ~feature_inventory.is_target, "column_name"].tolist(),
    "high_cardinality_categorical_features": feature_inventory.loc[(feature_inventory.cardinality_class == "High cardinality") & (feature_inventory.inferred_feature_type == "categorical"), "column_name"].tolist(),
    "possible_identifier_features": feature_inventory.loc[feature_inventory.possible_identifier, "column_name"].tolist(),
}
(ARTIFACT_DIRS["data_contract"] / "feature_sets.json").write_text(json.dumps(feature_sets, indent=2), encoding="utf-8")
display(feature_inventory[["column_name", "dataset_membership", "inferred_feature_type", "unique_values", "possible_identifier"]].head(10))

,column_name,dataset_membership,inferred_feature_type,unique_values,possible_identifier
0,loan_amount_000s,both,target,2245,False
1,respondent_id,both,categorical,4278,True
2,agency_name,both,categorical,6,False
3,loan_type_name,both,categorical,4,False
...,...,...,...,...,...
6,owner_occupancy_name,both,categorical,3,False
7,preapproval_name,both,categorical,3,False
8,msamd_name,both,categorical,409,True
9,state_name,both,categorical,52,False


The sensitive difference contains eight fields. Code and identifier-like fields are marked as categorical or suspicious for later pipeline decisions. They are not removed automatically.

## 7. Sensitive Feature Comparison

The non-sensitive dataset must be an aligned subset of the sensitive dataset. This is needed for fair model comparison later.

In [7]:
sensitive_comparison = {
    "with_sensitive_feature_count": len(features_with_sensitive),
    "without_sensitive_feature_count": len(features_without_sensitive),
    "sensitive_feature_count": len(sensitive_features),
    "sensitive_features": sensitive_features,
    "non_sensitive_is_ordered_subset": without_columns == [c for c in with_columns if c not in sensitive_features],
}
if not sensitive_comparison["non_sensitive_is_ordered_subset"]:
    raise ValueError("The non-sensitive schema is not the expected ordered subset.")
print(json.dumps(sensitive_comparison, indent=2))

{
  "with_sensitive_feature_count": 43,
  "without_sensitive_feature_count": 35,
  "sensitive_feature_count": 8,
  "sensitive_features": [
    "applicant_ethnicity_name",
    "co_applicant_ethnicity_name",
    "applicant_race_name_1",
    "co_applicant_race_name_1",
    "applicant_sex_name",
    "co_applicant_sex_name",
    "minority_population",
    "majority_minority_tract"
  ],
  "non_sensitive_is_ordered_subset": true
}


The non-sensitive schema is the exact ordered subset after eight sensitive fields are removed. The same `row_id` values can therefore be used for both modes.

## 8. Leakage and Suspicious-Column Checks

This scan checks exact target copies, target-like names, duplicate features, constants, accidental indexes, identifiers, and post-outcome names. Correlation alone is not treated as proof of leakage.

In [8]:
accidental_index_names = {"index", "level_0", "unnamed: 0", "row_id"}
exact_target_duplicates = []
numeric_target_diagnostics = []
for column in features_with_sensitive:
    series = df_with_sensitive_raw[column]
    if pd.api.types.is_numeric_dtype(series):
        values = series.to_numpy(dtype=float)
        target_values = y.to_numpy(dtype=float)
        finite = np.isfinite(values) & np.isfinite(target_values)
        exact_copy = bool(finite.all() and np.array_equal(values, target_values, equal_nan=True))
        if exact_copy:
            exact_target_duplicates.append(column)
        correlation = float(np.corrcoef(values[finite], target_values[finite])[0, 1]) if finite.sum() > 1 and np.std(values[finite]) > 0 else None
        affine_r_squared = None
        scaled_residual_ratio = None
        if finite.sum() > 1 and np.std(values[finite]) > 0:
            slope, intercept = np.polyfit(values[finite], target_values[finite], 1)
            fitted = slope * values[finite] + intercept
            residual = target_values[finite] - fitted
            denominator = np.sum((target_values[finite] - target_values[finite].mean()) ** 2)
            affine_r_squared = float(1 - np.sum(residual ** 2) / denominator) if denominator > 0 else None
            scaled_residual_ratio = float(np.max(np.abs(residual)) / max(np.ptp(target_values[finite]), 1.0))
        numeric_target_diagnostics.append({
            "column": column, "correlation_with_target": correlation,
            "affine_r_squared": affine_r_squared, "max_scaled_affine_residual": scaled_residual_ratio,
            "exact_target_copy": exact_copy,
            "near_affine_warning": bool(affine_r_squared is not None and affine_r_squared > 0.999999 and scaled_residual_ratio is not None and scaled_residual_ratio < 1e-6),
        })

fingerprints = {}
for column in features_with_sensitive:
    fingerprints[column] = int(pd.util.hash_pandas_object(df_with_sensitive_raw[column], index=False).sum())
duplicate_pairs = []
for i, left in enumerate(features_with_sensitive):
    for right in features_with_sensitive[i + 1:]:
        if fingerprints[left] == fingerprints[right] and semantically_equal(df_with_sensitive_raw[left], df_with_sensitive_raw[right]):
            duplicate_pairs.append((left, right))

leakage_rows = []
for column in features_with_sensitive:
    lower = column.lower()
    reasons = []
    status = "Safe"
    if column in exact_target_duplicates:
        status, reasons = "Confirmed leakage", ["exact target duplicate"]
    else:
        if lower in accidental_index_names: reasons.append("clear accidental or internal index")
        if TARGET.lower() in lower or "loan_amount" in lower: reasons.append("target-like column name")
        if feature_inventory.set_index("column_name").loc[column, "is_constant"]: reasons.append("constant column")
        if feature_inventory.set_index("column_name").loc[column, "possible_identifier"]: reasons.append("identifier or geographic code risk")
        if any(column in pair for pair in duplicate_pairs): reasons.append("duplicate feature content")
        if any(token in lower for token in ["loan_approved", "action_taken", "final_outcome"]): reasons.append("possible post-outcome name")
        if reasons: status = "Suspicious"
    leakage_rows.append({"column": column, "classification": status, "reasons": "; ".join(reasons) or "no rule-based warning"})

leakage_report = pd.DataFrame(leakage_rows)
target_diagnostic_frame = pd.DataFrame(numeric_target_diagnostics)
if not target_diagnostic_frame.empty:
    diagnostic_rows = target_diagnostic_frame.assign(
        classification=lambda x: np.where(x.near_affine_warning, "Strong leakage candidate", "Numeric target diagnostic"),
        reasons=lambda x: np.where(x.near_affine_warning, "almost exact affine target transform", "warning-only numeric comparison; correlation is not proof of leakage")
    )[["column", "classification", "reasons", "correlation_with_target", "affine_r_squared", "max_scaled_affine_residual"]]
    leakage_report = pd.concat([leakage_report, diagnostic_rows], ignore_index=True, sort=False)
leakage_report.to_csv(ARTIFACT_DIRS["data_contract"] / "leakage_and_suspicious_columns.csv", index=False)
auto_exclusions = sorted(set(exact_target_duplicates + [c for c in features_with_sensitive if c.lower() in accidental_index_names]))
print("Exact target duplicates:", exact_target_duplicates)
print("Duplicate feature pairs:", duplicate_pairs[:10])
print("Near-affine target warnings:", target_diagnostic_frame.loc[target_diagnostic_frame.near_affine_warning, "column"].tolist())
display(leakage_report[leakage_report.classification != "Safe"].head(12))

Exact target duplicates: []
Duplicate feature pairs: []
Near-affine target warnings: []


,column,classification,reasons,correlation_with_target,affine_r_squared,max_scaled_affine_residual
0,respondent_id,Suspicious,identifier or geographic code risk,NaN,NaN,NaN
7,msamd_name,Suspicious,identifier or geographic code risk,NaN,NaN,NaN
9,state_code,Suspicious,identifier or geographic code risk,NaN,NaN,NaN
10,county_name,Suspicious,identifier or geographic code risk,NaN,NaN,NaN
...,...,...,...,...,...,...
45,census_tract_number,Numeric target diagnostic,warning-only numeric comparison; correlation i...,-0.003194,0.000010,0.997506
46,applicant_income_000s,Numeric target diagnostic,warning-only numeric comparison; correlation i...,0.447505,0.200261,0.768228
47,population,Numeric target diagnostic,warning-only numeric comparison; correlation i...,-0.005248,0.000028,0.997512
48,minority_population,Numeric target diagnostic,warning-only numeric comparison; correlation i...,-0.007937,0.000063,0.997547


No feature is an exact target copy. Identifier and geography fields remain warning items for later deployment-focused tests. Engineered input transforms are redundant with raw inputs but are not target leakage.

## 9. Shared Train-Test Split

A zero-based `row_id` keeps the original row position without changing either source file. Target quantile bins are used only for stratification and are never model features.

In [9]:
# Stage 2 reuses the saved Stage 1 split. It never calls a new main splitter.
train_path = ARTIFACT_DIRS["splits"] / "train_row_ids.csv"
test_path = ARTIFACT_DIRS["splits"] / "test_row_ids.csv"
split_config_path = ARTIFACT_DIRS["splits"] / "split_config.json"
train_ids = pd.read_csv(train_path)["row_id"].to_numpy(dtype=np.int64)
test_ids = pd.read_csv(test_path)["row_id"].to_numpy(dtype=np.int64)
split_config = json.loads(split_config_path.read_text(encoding="utf-8"))
row_ids = np.arange(len(y), dtype=np.int64)
if len(np.intersect1d(train_ids, test_ids)) != 0:
    raise AssertionError("Saved train and test rows overlap.")
if not np.array_equal(np.sort(np.concatenate([train_ids, test_ids])), row_ids):
    raise AssertionError("Saved train and test rows do not cover all rows.")
print({"train_rows_loaded": len(train_ids), "test_rows_locked": len(test_ids), "split_recreated": False})

{'train_rows_loaded': 399788, 'test_rows_locked': 99948, 'split_recreated': False}


The shared split contains 399,788 training rows and 99,948 locked test rows. All ten requested target bins were valid.

## 10. Shared Cross-Validation Folds

Fold bins are created again from training targets only. One validation-fold number is saved for each training row.

In [10]:
# Stage 2 loads the saved fold assignment and does not create new folds.
cv_path = ARTIFACT_DIRS["splits"] / "cv_fold_assignments.csv"
cv_assignments = pd.read_csv(cv_path, dtype={"row_id": "int64", "fold": "int64"}).sort_values("row_id").reset_index(drop=True)
if cv_assignments["row_id"].duplicated().any():
    raise AssertionError("A training row has more than one saved fold.")
if not np.array_equal(cv_assignments["row_id"].to_numpy(), np.sort(train_ids)):
    raise AssertionError("Saved CV rows do not equal saved training rows.")
if set(cv_assignments["fold"]) != {0, 1, 2}:
    raise AssertionError("Saved CV fold labels are invalid.")
display(cv_assignments.groupby("fold").size().rename("rows").to_frame())

,rows
fold,
0,133263
1,133263
2,133262


Every training row has one validation-fold assignment. Fold sizes differ by at most one row, and no test row is included.

In [11]:
# This stage validates training folds only. Historical test statistics are not recomputed or used.
def distribution_summary(name, values):
    values = pd.Series(values)
    return {
        "set": name, "rows": len(values), "mean": float(values.mean()), "std": float(values.std()),
        "median": float(values.median()), "p90": float(values.quantile(.90)),
        "p95": float(values.quantile(.95)), "p99": float(values.quantile(.99)),
        "max": float(values.max()), "skew": float(values.skew()),
    }

y_array = y.to_numpy()
train_target = y_array[train_ids]
train_fold_lookup = cv_assignments.set_index("row_id")["fold"]
distribution_rows = [distribution_summary("train", train_target)]
for fold in range(CONFIG["n_cv_folds"]):
    fold_ids = cv_assignments.loc[cv_assignments["fold"] == fold, "row_id"].to_numpy()
    distribution_rows.append(distribution_summary(f"cv_fold_{fold}", y_array[fold_ids]))
distribution_table = pd.DataFrame(distribution_rows)
split_checks = {
    "train_test_overlap_zero": len(np.intersect1d(train_ids, test_ids)) == 0,
    "train_test_coverage_complete": np.array_equal(np.sort(np.concatenate([train_ids, test_ids])), row_ids),
    "cv_coverage_complete": np.array_equal(cv_assignments["row_id"].to_numpy(), np.sort(train_ids)),
    "no_test_row_in_cv": len(np.intersect1d(cv_assignments["row_id"].to_numpy(), test_ids)) == 0,
    "fold_sizes_reasonable": int(cv_assignments.groupby("fold").size().max() - cv_assignments.groupby("fold").size().min()) <= 1,
    "same_rows_for_both_sensitive_modes": len(df_with_sensitive_raw) == len(df_without_sensitive_raw) == len(row_ids),
    "target_order_correct": common_values_equal[TARGET],
}
if not all(split_checks.values()):
    raise AssertionError(f"Saved split verification failed: {split_checks}")
display(distribution_table)
print("Historical test statistics were not read or recomputed in Stage 2.")

,set,rows,mean,std,median,p90,p95,p99,max,skew
0,train,399788,247.346464,317.725921,197.0,444.0,599.0,1100.0,99000.0,122.066931
1,cv_fold_0,133263,246.905428,246.799595,197.0,444.0,599.0,1104.0,28000.0,16.593964
2,cv_fold_1,133263,247.755416,363.888181,197.0,444.0,600.0,1088.0,99000.0,152.825434
3,cv_fold_2,133262,247.378548,330.946900,197.0,444.0,599.0,1101.0,81000.0,114.009570


Historical test statistics were not read or recomputed in Stage 2.


Target quantiles are close across train, test, and folds. The test maximum is lower because the full data has a very rare extreme tail. This is reported as a risk; the test set is not used to change the split.

## 11. Regression Evaluation Metrics

Metrics are calculated on the original target scale. Negative predictions are kept for ordinary metrics. Only the clearly named clipped RMSLE uses zero-clipped predictions.

In [12]:
def _validated_numeric_vector(values, name):
    array = np.asarray(values)
    if array.ndim != 1: raise ValueError(f"{name} must be one-dimensional.")
    if array.size == 0: raise ValueError(f"{name} must not be empty.")
    try: array = array.astype(float)
    except (TypeError, ValueError) as exc: raise TypeError(f"{name} must contain numeric values.") from exc
    if not np.isfinite(array).all(): raise ValueError(f"{name} contains NaN or infinite values.")
    return array

def evaluate_regression_predictions(y_true, y_pred, unit_confirmed=True):
    true = _validated_numeric_vector(y_true, "y_true")
    pred = _validated_numeric_vector(y_pred, "y_pred")
    if len(true) != len(pred): raise ValueError("y_true and y_pred must have the same length.")
    errors = pred - true
    absolute_errors = np.abs(errors)
    mse = float(np.mean(errors ** 2))
    negative_rate = float(np.mean(pred < 0))
    metric_warnings = []
    if np.any(true == 0):
        mape = None
        metric_warnings.append("MAPE is unavailable because y_true contains zero.")
    else:
        mape = float(np.mean(np.abs(errors / true)) * 100)
    denominator = float(np.sum(np.abs(true)))
    wape = None if denominator == 0 else float(np.sum(absolute_errors) / denominator * 100)
    if denominator == 0: metric_warnings.append("WAPE is unavailable because sum(abs(y_true)) is zero.")
    r_squared = None if len(true) < 2 or np.all(true == true[0]) else float(r2_score(true, pred))
    if r_squared is None: metric_warnings.append("R-squared is unavailable for fewer than two values or a constant target.")
    rmsle = None
    rmsle_clipped_zero = None
    if np.any(true < 0):
        metric_warnings.append("RMSLE is unavailable because y_true contains negative values.")
    elif negative_rate > 0:
        rmsle_clipped_zero = float(np.sqrt(np.mean((np.log1p(np.clip(pred, 0, None)) - np.log1p(true)) ** 2)))
        metric_warnings.append("Negative predictions were clipped only for rmsle_clipped_zero.")
    else:
        rmsle = float(np.sqrt(np.mean((np.log1p(pred) - np.log1p(true)) ** 2)))
    mae = float(np.mean(absolute_errors))
    rmse = float(np.sqrt(mse))
    return {
        "mae": mae, "mse": mse, "rmse": rmse, "mape_percent": mape,
        "r_squared": r_squared, "rmsle": rmsle, "rmsle_clipped_zero": rmsle_clipped_zero,
        "median_absolute_error": float(np.median(absolute_errors)), "wape_percent": wape,
        "mean_signed_error": float(np.mean(errors)), "p90_absolute_error": float(np.quantile(absolute_errors, .90)),
        "negative_prediction_rate": negative_rate,
        "mae_usd": mae * 1000 if unit_confirmed else None,
        "rmse_usd": rmse * 1000 if unit_confirmed else None,
        "metric_warnings": metric_warnings,
    }

MAE is the primary metric. It is reported in thousands of dollars and US dollars. MAPE and WAPE are percentages, and mean signed error is positive for overprediction.

## 12. Metric Unit Tests

Small known examples check correct values and error handling. The notebook stops if any assertion fails.

In [13]:
perfect = evaluate_regression_predictions([1, 2, 3], [1, 2, 3])
assert perfect["mae"] == 0 and perfect["mse"] == 0 and perfect["rmse"] == 0
assert perfect["r_squared"] == 1

known = evaluate_regression_predictions([1, 2, 3], [2, 2, 1])
assert math.isclose(known["mae"], 1.0)
assert math.isclose(known["mse"], 5 / 3)
assert math.isclose(known["rmse"], math.sqrt(5 / 3))
assert math.isclose(known["mean_signed_error"], -1 / 3)

true_input = np.array([1., 2., 3.]); negative_input = np.array([-1., 2., 3.])
negative_case = evaluate_regression_predictions(true_input, negative_input)
assert math.isclose(negative_case["negative_prediction_rate"], 1 / 3)
assert negative_case["rmsle"] is None and negative_case["rmsle_clipped_zero"] is not None
assert np.array_equal(negative_input, np.array([-1., 2., 3.]))

zero_case = evaluate_regression_predictions([0, 0], [0, 1])
assert zero_case["mape_percent"] is None and zero_case["wape_percent"] is None
assert evaluate_regression_predictions([2, 2], [2, 3])["r_squared"] is None

for bad_true, bad_pred, expected_error in [
    ([1, 2], [1], ValueError), ([1, 2], [1, np.nan], ValueError),
    ([1, 2], [1, np.inf], ValueError), ([1, "x"], [1, 2], TypeError),
    ([], [], ValueError), ([[1, 2]], [[1, 2]], ValueError),
]:
    try:
        evaluate_regression_predictions(bad_true, bad_pred)
        raise AssertionError("Invalid metric input did not raise an error.")
    except expected_error:
        pass

metric_schema = {
    "primary_metric": "mae", "target_unit": CONFIG["target_unit"],
    "definitions": {
        "mae": "mean absolute error in target units", "mse": "mean squared error",
        "rmse": "square root of MSE in target units", "mape_percent": "mean absolute percentage error; unavailable for zero targets",
        "r_squared": "coefficient of determination; unavailable for constant targets or fewer than two rows",
        "rmsle": "RMSLE when targets and predictions are non-negative", "rmsle_clipped_zero": "RMSLE with negative predictions clipped only for this metric",
        "median_absolute_error": "median absolute error", "wape_percent": "sum absolute error divided by sum absolute target, as percent",
        "mean_signed_error": "mean(prediction - target); positive means overprediction", "p90_absolute_error": "90th percentile absolute error",
        "negative_prediction_rate": "share of predictions below zero", "mae_usd": "MAE multiplied by 1000",
        "rmse_usd": "RMSE multiplied by 1000"
    },
    "all_final_metrics_use_original_target_scale": True,
}
(ARTIFACT_DIRS["data_contract"] / "metric_schema.json").write_text(json.dumps(metric_schema, indent=2), encoding="utf-8")
metric_unit_tests_passed = True
print("Metric unit tests passed:", metric_unit_tests_passed)

Metric unit tests passed: True


All metric tests passed. Invalid, missing, infinite, empty, and mismatched inputs raise clear errors.

## 13. Experiment Result Registry

The registry gives later prompts one stable result format. Existing valid rows are preserved.

In [14]:
REGISTRY_COLUMNS = [
    "experiment_id", "timestamp_utc", "model_family", "model_name", "sensitive_mode", "feature_set",
    "target_mode", "evaluation_stage", "fold_number", "training_row_count", "validation_row_count", "test_row_count",
    "parameter_json", "mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero",
    "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error", "negative_prediction_rate",
    "fit_time_seconds", "prediction_time_seconds", "status", "notes", "model_artifact_path", "prediction_artifact_path"
]
registry_path = ARTIFACT_DIRS["results"] / "experiment_results.csv"

def validate_result_registry(frame):
    missing = [c for c in REGISTRY_COLUMNS if c not in frame.columns]
    extra = [c for c in frame.columns if c not in REGISTRY_COLUMNS]
    if missing or extra: raise ValueError(f"Registry schema mismatch. Missing={missing}, extra={extra}")
    if frame["experiment_id"].dropna().duplicated().any(): raise ValueError("Experiment IDs must be unique.")
    if not frame.empty:
        if frame["experiment_id"].isna().any(): raise ValueError("Experiment IDs must not be missing.")
        if pd.to_datetime(frame["timestamp_utc"], errors="coerce", utc=True).isna().any(): raise ValueError("Registry timestamps must be valid UTC timestamps.")
        if (~frame["status"].isin(["success", "failed", "skipped"])).any(): raise ValueError("Registry status is invalid.")
        for count_column in ["training_row_count", "validation_row_count", "test_row_count"]:
            values = pd.to_numeric(frame[count_column], errors="coerce")
            if values.isna().any() or (values < 0).any(): raise ValueError(f"{count_column} must be a non-negative number.")
        for value in frame["parameter_json"]:
            json.loads(value)
        metric_columns = ["mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero", "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error", "negative_prediction_rate"]
        numeric_metrics = frame[metric_columns].apply(pd.to_numeric, errors="coerce")
        if np.isinf(numeric_metrics.to_numpy(dtype=float)).any(): raise ValueError("Registry metrics must not be infinite.")
    return True

def append_experiment_result(frame, record):
    missing = [c for c in REGISTRY_COLUMNS if c not in record]
    extra = [c for c in record if c not in REGISTRY_COLUMNS]
    if missing or extra: raise ValueError(f"Result record schema mismatch. Missing={missing}, extra={extra}")
    result = pd.concat([frame, pd.DataFrame([record])], ignore_index=True)
    validate_result_registry(result)
    return result

def save_result_registry(frame, path=registry_path):
    validate_result_registry(frame)
    temp_path = path.with_suffix(".tmp")
    frame.to_csv(temp_path, index=False)
    temp_path.replace(path)

if registry_path.exists():
    experiment_results = pd.read_csv(registry_path)
    validate_result_registry(experiment_results)
else:
    experiment_results = pd.DataFrame(columns=REGISTRY_COLUMNS)
save_result_registry(experiment_results)
reloaded_registry = pd.read_csv(registry_path)
registry_round_trip_passed = validate_result_registry(reloaded_registry) and list(reloaded_registry.columns) == REGISTRY_COLUMNS
synthetic_record = {column: None for column in REGISTRY_COLUMNS}
synthetic_record.update({
    "experiment_id": "registry_test_only", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_family": "test", "model_name": "test", "sensitive_mode": "without_sensitive",
    "feature_set": "test", "target_mode": "raw", "evaluation_stage": "unit_test", "fold_number": 0,
    "training_row_count": 2, "validation_row_count": 1, "test_row_count": 0, "parameter_json": "{}",
    "status": "success", "notes": "in-memory registry validation only"
})
synthetic_registry = append_experiment_result(pd.DataFrame(columns=REGISTRY_COLUMNS), synthetic_record)
registry_nonempty_append_test_passed = validate_result_registry(synthetic_registry)
print({"registry_rows": len(reloaded_registry), "round_trip_passed": registry_round_trip_passed,
       "synthetic_append_test_passed": registry_nonempty_append_test_passed})

{'registry_rows': 107, 'round_trip_passed': True, 'synthetic_append_test_passed': True}


The registry saves, reloads, and validates correctly. Stage 1 added no model rows. A later stage may add rows with deterministic IDs.

## 14. Saved Artifacts

This section checks source safety again and confirms that required artifacts exist.

In [15]:
source_hashes_after = {name: file_fingerprint(path) for name, path in protected_source_paths.items()}
after_path = ARTIFACT_DIRS["data_contract"] / "source_hashes_after.json"
if not after_path.exists():
    after_path.write_text(json.dumps(source_hashes_after, indent=2), encoding="utf-8")
hashes_unchanged = all(source_hashes_before[name]["sha256"] == source_hashes_after[name]["sha256"] for name in protected_source_paths)
if not hashes_unchanged:
    raise RuntimeError("CRITICAL: A protected Stage 1 source hash changed.")

required_artifacts = [
    before_path, after_path,
    ARTIFACT_DIRS["data_contract"] / "feature_inventory.csv",
    ARTIFACT_DIRS["data_contract"] / "feature_sets.json",
    ARTIFACT_DIRS["data_contract"] / "leakage_and_suspicious_columns.csv",
    ARTIFACT_DIRS["data_contract"] / "metric_schema.json",
    ARTIFACT_DIRS["splits"] / "train_row_ids.csv",
    ARTIFACT_DIRS["splits"] / "test_row_ids.csv",
    ARTIFACT_DIRS["splits"] / "split_config.json",
    ARTIFACT_DIRS["splits"] / "cv_fold_assignments.csv",
    ARTIFACT_DIRS["splits"] / "split_verification.json",
    registry_path,
]
required_artifacts_exist = all(path.is_file() and path.stat().st_size > 0 for path in required_artifacts)
artifact_table = pd.DataFrame({"artifact": [str(p.relative_to(PROJECT_ROOT)) for p in required_artifacts], "exists": [p.is_file() for p in required_artifacts]})
display(artifact_table)
print("Protected Stage 1 source hashes unchanged:", hashes_unchanged)

,artifact,exists
0,artifacts\data_contract\source_hashes_before.json,True
1,artifacts\data_contract\source_hashes_after.json,True
2,artifacts\data_contract\feature_inventory.csv,True
3,artifacts\data_contract\feature_sets.json,True
...,...,...
8,artifacts\splits\split_config.json,True
9,artifacts\splits\cv_fold_assignments.csv,True
10,artifacts\splits\split_verification.json,True
11,artifacts\results\experiment_results.csv,True


Protected Stage 1 source hashes unchanged: True


Required data-contract, split, metric, and registry artifacts exist. Both source CSV hashes are unchanged.

## 15. Verification Summary

The final report stores real pass or fail values. Reaching this cell proves that all earlier notebook assertions passed.

In [16]:
# Stage 1 completion is historical. Later stages preserve its saved PASS report.
verification_path = ARTIFACT_DIRS["reports"] / "prompt1_verification.json"
prompt1_historical_report = json.loads(verification_path.read_text(encoding="utf-8"))
prompt1_historical_checks = {
    "saved_prompt1_status_pass": prompt1_historical_report.get("status") == "PASS",
    "prompt1_recorded_no_model_training": bool(prompt1_historical_report.get("checks", {}).get("no_real_model_trained", False)),
    "current_source_hashes_match_prompt1": all(
        file_fingerprint(protected_source_paths[name])["sha256"] == source_hashes_before[name]["sha256"]
        for name in protected_source_paths
    ),
    "saved_split_still_valid": all(split_checks.values()),
    "metric_unit_tests_still_pass": metric_unit_tests_passed,
    "registry_schema_still_valid": registry_round_trip_passed and registry_nonempty_append_test_passed,
}
if not all(prompt1_historical_checks.values()):
    raise AssertionError(f"Stage 1 historical validation failed: {prompt1_historical_checks}")
display(pd.DataFrame.from_dict(prompt1_historical_checks, orient="index", columns=["passed"]))
print("STAGE 1 HISTORICAL STATUS: PASS (report preserved)")

,passed
saved_prompt1_status_pass,True
prompt1_recorded_no_model_training,True
current_source_hashes_match_prompt1,True
saved_split_still_valid,True
metric_unit_tests_still_pass,True
registry_schema_still_valid,True


STAGE 1 HISTORICAL STATUS: PASS (report preserved)


### Stage 1 Historical Completion Note

Stage 1 created and verified the shared data foundation. It trained no model at that time. Later stages reuse the locked split, folds, metrics, and historical PASS report.

## 16. Stage 2 Objective and Rules

Stage 2 compares six linear-family models in two sensitive modes. It uses training rows and saved folds only. The locked test set is not used for fitting, prediction, selection, or statistics.

Internal artifact IDs keep their old technical names to preserve reproducibility.

In [17]:
import gc
import os
import platform
import subprocess
import sys
import time
import warnings
from copy import deepcopy

import joblib
import matplotlib.pyplot as plt
import nbformat as nbf
import sklearn
from IPython.display import Markdown
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split

from prompt2_pipeline_utils import (
    canonical_json, configuration_digest, deterministic_experiment_id,
    fitted_estimator, fitted_pipeline, make_complete_pipeline,
    make_preprocessor, transformed_feature_names,
)

PROMPT2_VERSION = "linear_compact_v1"
P2_DIRS = {
    "models": CONFIG["artifact_root"] / "models" / "linear",
    "predictions": CONFIG["artifact_root"] / "predictions" / "linear",
    "results": CONFIG["artifact_root"] / "results" / "prompt2",
    "features": CONFIG["artifact_root"] / "features" / "linear",
    "coefficients": CONFIG["artifact_root"] / "features" / "linear" / "coefficients",
    "figures": CONFIG["artifact_root"] / "figures" / "prompt2",
    "reports": CONFIG["artifact_root"] / "reports",
    "manifests": CONFIG["artifact_root"] / "manifests",
}
for directory in P2_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)
P2_START_TIME = time.perf_counter()
print("Stage 2 directories are ready. Large fits will run sequentially.")

Stage 2 directories are ready. Large fits will run sequentially.


## 17. Stage 1 Artifact Validation

Stage 2 checks live source and split hashes against a saved baseline. It also reloads row IDs, folds, feature metadata, metrics, and the registry from disk.

In [18]:
def file_sha256(path):
    return sha256_stream(Path(path))

prompt2_protected_paths = {
    "with_sensitive_csv": CONFIG["with_sensitive_path"],
    "without_sensitive_csv": CONFIG["without_sensitive_path"],
    "part1_notebook": CONFIG["source_notebook_path"],
    "train_row_ids": ARTIFACT_DIRS["splits"] / "train_row_ids.csv",
    "test_row_ids": ARTIFACT_DIRS["splits"] / "test_row_ids.csv",
    "cv_fold_assignments": ARTIFACT_DIRS["splits"] / "cv_fold_assignments.csv",
    "split_config": ARTIFACT_DIRS["splits"] / "split_config.json",
}
protected_before_path = P2_DIRS["manifests"] / "prompt2_protected_hashes_before.json"
current_protected_hashes = {name: file_sha256(path) for name, path in prompt2_protected_paths.items()}
if protected_before_path.exists():
    prompt2_protected_hashes_before = json.loads(protected_before_path.read_text(encoding="utf-8"))
    if prompt2_protected_hashes_before != current_protected_hashes:
        raise RuntimeError("A protected Stage 1 source or split artifact changed before Stage 2.")
else:
    prompt2_protected_hashes_before = current_protected_hashes
    protected_before_path.write_text(json.dumps(prompt2_protected_hashes_before, indent=2), encoding="utf-8")

disk_train_ids = pd.read_csv(prompt2_protected_paths["train_row_ids"])["row_id"].to_numpy(dtype=np.int64)
disk_test_ids = pd.read_csv(prompt2_protected_paths["test_row_ids"])["row_id"].to_numpy(dtype=np.int64)
disk_cv = pd.read_csv(prompt2_protected_paths["cv_fold_assignments"], dtype={"row_id": "int64", "fold": "int64"}).sort_values("row_id").reset_index(drop=True)
disk_feature_sets = json.loads((ARTIFACT_DIRS["data_contract"] / "feature_sets.json").read_text(encoding="utf-8"))
disk_metric_schema = json.loads((ARTIFACT_DIRS["data_contract"] / "metric_schema.json").read_text(encoding="utf-8"))
disk_registry = pd.read_csv(registry_path)
prompt1_report = json.loads((ARTIFACT_DIRS["reports"] / "prompt1_verification.json").read_text(encoding="utf-8"))
artifact_checks = {
    "prompt1_report_pass": prompt1_report.get("status") == "PASS",
    "train_ids_match_memory": np.array_equal(disk_train_ids, train_ids),
    "test_ids_match_memory": np.array_equal(disk_test_ids, test_ids),
    "cv_matches_memory": disk_cv.equals(cv_assignments),
    "target_matches_metadata": disk_feature_sets["target_column"] == TARGET,
    "sensitive_features_match": disk_feature_sets["sensitive_features"] == sensitive_features,
    "metric_schema_primary_mae": disk_metric_schema["primary_metric"] == "mae",
    "registry_schema_valid": validate_result_registry(disk_registry),
    "train_test_overlap_zero": len(np.intersect1d(disk_train_ids, disk_test_ids)) == 0,
    "cv_training_only": len(np.intersect1d(disk_cv["row_id"].to_numpy(), disk_test_ids)) == 0,
    "target_positive_on_training": bool((y_array[disk_train_ids] > 0).all()),
}
if not all(artifact_checks.values()):
    raise AssertionError(f"Stage 1 artifact validation failed: {artifact_checks}")
display(pd.DataFrame.from_dict(artifact_checks, orient="index", columns=["passed"]))

,passed
prompt1_report_pass,True
train_ids_match_memory,True
test_ids_match_memory,True
cv_matches_memory,True
...,...
registry_schema_valid,True
train_test_overlap_zero,True
cv_training_only,True
target_positive_on_training,True


Stage 1 artifacts match memory and their baseline hashes. The test IDs are used only in exclusion checks. No test target or test feature view is created.

## 18. Training Data Views

Training views follow the saved row order. This makes OOF positions and original `row_id` values easy to audit.

In [19]:
train_ids = np.sort(disk_train_ids)
test_id_set = set(disk_test_ids.tolist())
fold_by_train_position = disk_cv.set_index("row_id").loc[train_ids, "fold"].to_numpy(dtype=int)
y_train_prompt2 = y_array[train_ids].astype(float, copy=True)
if len(y_train_prompt2) != len(train_ids) or not np.isfinite(y_train_prompt2).all():
    raise AssertionError("Stage 2 training target view is invalid.")
print({"training_rows": len(train_ids), "folds": pd.Series(fold_by_train_position).value_counts().sort_index().to_dict(), "test_rows_used": 0})

{'training_rows': 399788, 'folds': {0: 133263, 1: 133263, 2: 133262}, 'test_rows_used': 0}


The training view contains 399,788 rows and the saved fold sizes. The locked test rows are not part of any model view.

## 19. Linear Feature Pack

The compact pack removes detailed identifiers and one exact duplicate numeric field. It keeps useful numeric, categorical, and existing engineered features.

In [20]:
linear_exclusions = {
    "respondent_id": "high-cardinality lender identifier",
    "msamd_name": "high-cardinality detailed metro geography",
    "county_name": "high-cardinality detailed county geography",
    "county_code": "geographic code; readable broader geography is retained",
    "state_code": "redundant code for retained state_name",
    "census_tract_number": "high-cardinality tract identifier",
    "tract_to_msamd_income": "exact linear duplicate of retained tract_income_ratio times 100",
}
base_without = list(disk_feature_sets["features_without_sensitive"])
linear_compact_without_sensitive = [feature for feature in base_without if feature not in linear_exclusions]
linear_compact_with_sensitive = linear_compact_without_sensitive + list(sensitive_features)
difference = set(linear_compact_with_sensitive) - set(linear_compact_without_sensitive)
if difference != set(sensitive_features):
    raise AssertionError("Linear feature-pack difference does not equal the validated sensitive set.")
if linear_compact_with_sensitive[:len(linear_compact_without_sensitive)] != linear_compact_without_sensitive:
    raise AssertionError("The common feature order changed.")
if TARGET in linear_compact_with_sensitive or "row_id" in linear_compact_with_sensitive:
    raise AssertionError("Target or row_id entered a feature pack.")

inventory_lookup = feature_inventory.set_index("column_name")
numeric_without = [f for f in linear_compact_without_sensitive if inventory_lookup.loc[f, "inferred_feature_type"] == "numeric"]
categorical_without = [f for f in linear_compact_without_sensitive if f not in numeric_without]
numeric_with = [f for f in linear_compact_with_sensitive if inventory_lookup.loc[f, "inferred_feature_type"] == "numeric"]
categorical_with = [f for f in linear_compact_with_sensitive if f not in numeric_with]
linear_feature_sets = {
    "version": PROMPT2_VERSION,
    "without_sensitive": linear_compact_without_sensitive,
    "with_sensitive": linear_compact_with_sensitive,
    "numeric_without_sensitive": numeric_without,
    "categorical_without_sensitive": categorical_without,
    "numeric_with_sensitive": numeric_with,
    "categorical_with_sensitive": categorical_with,
    "validated_sensitive_features": sensitive_features,
    "exclusions": linear_exclusions,
}
feature_set_path = P2_DIRS["features"] / "linear_feature_sets.json"
feature_set_path.write_text(json.dumps(linear_feature_sets, indent=2), encoding="utf-8")

feature_inventory_rows = []
for feature in features_with_sensitive:
    included = feature in linear_compact_with_sensitive
    feature_inventory_rows.append({
        "feature": feature,
        "feature_type": inventory_lookup.loc[feature, "inferred_feature_type"],
        "included": included,
        "sensitive": feature in sensitive_features,
        "exclusion_reason": "" if included else linear_exclusions.get(feature, "not in compact family pack"),
    })
linear_feature_inventory = pd.DataFrame(feature_inventory_rows)
linear_feature_inventory.to_csv(P2_DIRS["features"] / "linear_feature_inventory.csv", index=False)

X_without_train = df_without_sensitive_raw.iloc[train_ids][linear_compact_without_sensitive].copy()
X_with_train = df_with_sensitive_raw.iloc[train_ids][linear_compact_with_sensitive].copy()
X_without_train.index = np.arange(len(X_without_train))
X_with_train.index = np.arange(len(X_with_train))
print({"without_sensitive_features": len(linear_compact_without_sensitive), "with_sensitive_features": len(linear_compact_with_sensitive),
       "numeric_without": len(numeric_without), "categorical_without": len(categorical_without)})
display(linear_feature_inventory.loc[~linear_feature_inventory["included"]].head(10))

{'without_sensitive_features': 28, 'with_sensitive_features': 36, 'numeric_without': 16, 'categorical_without': 12}


,feature,feature_type,included,sensitive,exclusion_reason
0,respondent_id,categorical,False,False,high-cardinality lender identifier
7,msamd_name,categorical,False,False,high-cardinality detailed metro geography
9,state_code,categorical,False,False,redundant code for retained state_name
10,county_name,categorical,False,False,high-cardinality detailed county geography
11,county_code,categorical,False,False,geographic code; readable broader geography is...
12,census_tract_number,categorical,False,False,high-cardinality tract identifier
24,tract_to_msamd_income,numeric,False,False,exact linear duplicate of retained tract_incom...


The non-sensitive pack has 28 fields. The sensitive pack has 36 fields and adds exactly the eight validated sensitive fields. Detailed lender and geography identifiers are excluded to control one-hot size.

## 20. Shared Linear Preprocessing

Each complete pipeline gets a new preprocessor. Numeric values use median imputation and the selected scaler. Categories use safe conversion, most-frequent imputation, rare-level grouping, and sparse one-hot encoding.

In [21]:
preprocessor_definition = make_preprocessor(numeric_without, categorical_without, "standard")
preprocessing_checks = {
    "numeric_has_imputer_and_scaler": [name for name, _ in preprocessor_definition.transformers[0][1].steps] == ["imputer", "scaler"],
    "categorical_has_conversion_imputer_encoder": [name for name, _ in preprocessor_definition.transformers[1][1].steps] == ["to_object", "imputer", "encoder"],
    "sparse_output_forced": preprocessor_definition.sparse_threshold == 1.0,
    "no_global_fit": not hasattr(preprocessor_definition, "transformers_"),
}
if not all(preprocessing_checks.values()):
    raise AssertionError(f"Preprocessor definition is invalid: {preprocessing_checks}")
del preprocessor_definition
print(preprocessing_checks)

{'numeric_has_imputer_and_scaler': True, 'categorical_has_conversion_imputer_encoder': True, 'sparse_output_forced': True, 'no_global_fit': True}


The factory is valid and remains unfitted. Learned imputation, scaling, and encoding will happen inside each candidate, fold, and final pipeline.

## 21. Pipeline and Experiment Utilities

These helpers create fresh pipelines, capture warnings, evaluate original-scale predictions, and upsert deterministic Stage 2 registry rows.

In [22]:
warning_records = []

def model_diagnostics(model):
    names = transformed_feature_names(model)
    estimator = fitted_estimator(model)
    coefficients = getattr(estimator, "coef_", None)
    nonzero = None if coefficients is None else int(np.count_nonzero(np.asarray(coefficients)))
    return {"transformed_features": len(names), "nonzero_coefficients": nonzero}

def fit_with_retries(configuration, X_fit, y_fit, numeric_features, categorical_features, context):
    model_name = configuration["model_name"]
    if model_name in {"lasso", "elastic_net"}:
        attempts = [{}, {"max_iter": 10000}, {"max_iter": 10000, "tol": 1e-3}]
    elif model_name == "gamma_regressor":
        attempts = [{}, {"max_iter": 1000}]
    else:
        attempts = [{}]
    last_model = None
    total_fit_time = 0.0
    for retry_number, overrides in enumerate(attempts):
        candidate = make_complete_pipeline(configuration, numeric_features, categorical_features, fit_overrides=overrides)
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            candidate.fit(X_fit, y_fit)
        elapsed = time.perf_counter() - start
        total_fit_time += elapsed
        convergence_warning = False
        for item in caught:
            warning_records.append({
                **context, "warning_class": item.category.__name__, "warning_message": str(item.message),
                "retry_number": retry_number, "fit_overrides": canonical_json(overrides),
            })
            convergence_warning = convergence_warning or issubclass(item.category, ConvergenceWarning)
        last_model = candidate
        if not convergence_warning:
            return candidate, total_fit_time, retry_number, "converged"
        del candidate
        gc.collect()
    raise RuntimeError(f"{model_name} did not converge after {len(attempts)} meaningful attempts: {context}")

def predict_and_measure(model, X_predict):
    start = time.perf_counter()
    prediction = np.asarray(model.predict(X_predict), dtype=float)
    elapsed = time.perf_counter() - start
    if not np.isfinite(prediction).all():
        raise ValueError("A model produced non-finite predictions.")
    return prediction, elapsed

def registry_record(experiment_id, configuration, sensitive_mode, stage, fold_number,
                    train_count, validation_count, metrics, fit_time, prediction_time,
                    status="success", notes="", model_path="", prediction_path=""):
    values = metrics or {}
    return {
        "experiment_id": experiment_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_family": "linear_family", "model_name": configuration["model_name"],
        "sensitive_mode": sensitive_mode, "feature_set": PROMPT2_VERSION,
        "target_mode": configuration["target_mode"], "evaluation_stage": stage,
        "fold_number": fold_number, "training_row_count": train_count,
        "validation_row_count": validation_count, "test_row_count": 0,
        "parameter_json": canonical_json(configuration),
        "mae": values.get("mae"), "mse": values.get("mse"), "rmse": values.get("rmse"),
        "mape_percent": values.get("mape_percent"), "r_squared": values.get("r_squared"),
        "rmsle": values.get("rmsle"), "rmsle_clipped_zero": values.get("rmsle_clipped_zero"),
        "median_absolute_error": values.get("median_absolute_error"), "wape_percent": values.get("wape_percent"),
        "mean_signed_error": values.get("mean_signed_error"), "p90_absolute_error": values.get("p90_absolute_error"),
        "negative_prediction_rate": values.get("negative_prediction_rate"),
        "fit_time_seconds": fit_time, "prediction_time_seconds": prediction_time,
        "status": status, "notes": notes, "model_artifact_path": model_path,
        "prediction_artifact_path": prediction_path,
    }

prompt2_registry_records = []

def upsert_prompt2_registry(base_frame, records):
    incoming = pd.DataFrame(records, columns=REGISTRY_COLUMNS)
    if incoming["experiment_id"].duplicated().any():
        raise ValueError("Incoming Stage 2 experiment IDs are not unique.")
    existing = base_frame.copy()
    old_timestamps = existing.set_index("experiment_id")["timestamp_utc"].to_dict() if not existing.empty else {}
    incoming["timestamp_utc"] = [old_timestamps.get(row.experiment_id, row.timestamp_utc) for row in incoming.itertuples()]
    kept = existing.loc[~existing["experiment_id"].isin(incoming["experiment_id"])].copy()
    combined = pd.concat([kept, incoming], ignore_index=True)
    validate_result_registry(combined)
    save_result_registry(combined)
    reloaded = pd.read_csv(registry_path)
    validate_result_registry(reloaded)
    if reloaded["experiment_id"].duplicated().any():
        raise AssertionError("Registry upsert created duplicate IDs.")
    return reloaded

id_test_config = {"model_name": "ridge", "target_mode": "raw", "scaler": "standard", "alpha": 1.0}
id_a = deterministic_experiment_id("ridge", "without_sensitive", "raw", "unit", None, id_test_config)
id_b = deterministic_experiment_id("ridge", "without_sensitive", "raw", "unit", None, dict(reversed(list(id_test_config.items()))))
id_c = deterministic_experiment_id("ridge", "without_sensitive", "raw", "unit", None, {**id_test_config, "alpha": 10.0})
assert id_a == id_b and id_a != id_c
print("Pipeline, warning, metric, deterministic ID, and registry-upsert utilities are ready.")

Pipeline, warning, metric, deterministic ID, and registry-upsert utilities are ready.


## 22. Development Screening Design

Fold 0 supplies validation rows. Folds 1 and 2 supply training rows. Target-stratified caps create one fixed sample for every candidate.

In [23]:
def stratified_cap(source_positions, cap, seed):
    source_positions = np.asarray(source_positions, dtype=int)
    values = y_train_prompt2[source_positions]
    bins = pd.qcut(values, q=10, labels=False, duplicates="drop").astype(int)
    if len(source_positions) <= cap:
        return np.sort(source_positions), bins
    selected, _ = train_test_split(
        source_positions, train_size=cap, random_state=seed, stratify=bins
    )
    selected = np.sort(selected)
    selected_bins = pd.qcut(y_train_prompt2[selected], q=10, labels=False, duplicates="drop").astype(int)
    return selected, selected_bins

development_train_source = np.flatnonzero(np.isin(fold_by_train_position, [1, 2]))
development_validation_source = np.flatnonzero(fold_by_train_position == 0)
dev_train_positions, dev_train_bins = stratified_cap(development_train_source, 80000, CONFIG["random_state"])
dev_validation_positions, dev_validation_bins = stratified_cap(development_validation_source, 20000, CONFIG["random_state"])
development_manifest = pd.concat([
    pd.DataFrame({"row_id": train_ids[dev_train_positions], "development_role": "train",
                  "original_cv_fold": fold_by_train_position[dev_train_positions], "target_bin": dev_train_bins}),
    pd.DataFrame({"row_id": train_ids[dev_validation_positions], "development_role": "validation",
                  "original_cv_fold": fold_by_train_position[dev_validation_positions], "target_bin": dev_validation_bins}),
], ignore_index=True).sort_values(["development_role", "row_id"]).reset_index(drop=True)
development_manifest_path = ARTIFACT_DIRS["splits"] / "prompt2_development_sample.csv"
development_manifest.to_csv(development_manifest_path, index=False)
development_manifest_hash = file_sha256(development_manifest_path)
development_checks = {
    "rows_unique": not development_manifest["row_id"].duplicated().any(),
    "no_test_rows": len(set(development_manifest["row_id"]) & test_id_set) == 0,
    "training_cap": int((development_manifest["development_role"] == "train").sum()) == 80000,
    "validation_cap": int((development_manifest["development_role"] == "validation").sum()) == 20000,
    "roles_follow_saved_folds": bool((development_manifest.loc[development_manifest.development_role == "validation", "original_cv_fold"] == 0).all()
                                    and development_manifest.loc[development_manifest.development_role == "train", "original_cv_fold"].isin([1, 2]).all()),
}
if not all(development_checks.values()):
    raise AssertionError(f"Development manifest failed: {development_checks}")
X_dev_train = X_without_train.iloc[dev_train_positions]
X_dev_validation = X_without_train.iloc[dev_validation_positions]
y_dev_train = y_train_prompt2[dev_train_positions]
y_dev_validation = y_train_prompt2[dev_validation_positions]
print({"development_train": len(dev_train_positions), "development_validation": len(dev_validation_positions),
       "manifest_sha256": development_manifest_hash})

{'development_train': 80000, 'development_validation': 20000, 'manifest_sha256': 'ace1d076e18310403f5e5bc83170ef67fcd6a9acd03a22c17bfeef5ad48873d3'}


The development design uses 80,000 training rows and 20,000 validation rows. All rows come from the saved training set, and the same manifest is used for every candidate.

## 23. Scaler Selection

A Ridge baseline compares Standard and Robust scaling with raw and log targets. The selected scaler is then fixed for every model family.

In [24]:
screening_results_path = P2_DIRS["results"] / "development_screening_results.csv"
selected_configurations_path = P2_DIRS["results"] / "selected_linear_configurations.json"

def candidate_key(configuration, group):
    return f"{group}__{configuration_digest(configuration)}"

scaler_candidates = [
    {"model_name": "ridge", "target_mode": target_mode, "scaler": scaler, "alpha": 1.0}
    for scaler in ["standard", "robust"] for target_mode in ["raw", "log1p"]
]

def model_candidate_grid(selected_scaler):
    candidates = {
        "dummy_median": [{"model_name": "dummy_median", "target_mode": "raw", "scaler": selected_scaler}],
        "linear_regression": [{"model_name": "linear_regression", "target_mode": mode, "scaler": selected_scaler} for mode in ["raw", "log1p"]],
        "ridge": [{"model_name": "ridge", "target_mode": mode, "scaler": selected_scaler, "alpha": alpha}
                  for mode in ["raw", "log1p"] for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]],
        "lasso": ([{"model_name": "lasso", "target_mode": "raw", "scaler": selected_scaler, "alpha": alpha, "max_iter": 5000, "tol": 1e-4}
                   for alpha in [0.01, 0.1, 1.0, 10.0]]
                  + [{"model_name": "lasso", "target_mode": "log1p", "scaler": selected_scaler, "alpha": alpha, "max_iter": 5000, "tol": 1e-4}
                     for alpha in [0.00001, 0.0001, 0.001, 0.01]]),
        "elastic_net": ([{"model_name": "elastic_net", "target_mode": "raw", "scaler": selected_scaler, "alpha": alpha, "l1_ratio": ratio, "max_iter": 5000, "tol": 1e-4}
                         for alpha in [0.01, 0.1, 1.0] for ratio in [0.2, 0.5, 0.8]]
                        + [{"model_name": "elastic_net", "target_mode": "log1p", "scaler": selected_scaler, "alpha": alpha, "l1_ratio": ratio, "max_iter": 5000, "tol": 1e-4}
                           for alpha in [0.00001, 0.0001, 0.001] for ratio in [0.2, 0.5, 0.8]]),
        "gamma_regressor": [{"model_name": "gamma_regressor", "target_mode": "raw", "scaler": selected_scaler,
                              "alpha": alpha, "max_iter": 500, "tol": 1e-5} for alpha in [0.0, 0.01, 0.1, 1.0]],
    }
    return candidates

screening_cache_valid = False
if screening_results_path.exists() and selected_configurations_path.exists():
    development_screening_results = pd.read_csv(screening_results_path)
    selected_linear_configurations = json.loads(selected_configurations_path.read_text(encoding="utf-8"))
    screening_cache_valid = (
        selected_linear_configurations.get("development_manifest_sha256") == development_manifest_hash
        and selected_linear_configurations.get("version") == PROMPT2_VERSION
        and set(selected_linear_configurations.get("models", {})) == {"dummy_median", "linear_regression", "ridge", "lasso", "elastic_net", "gamma_regressor"}
        and len(development_screening_results) == 47
        and development_screening_results["status"].isin(["success", "failed"]).all()
    )

if not screening_cache_valid:
    screening_rows = []
    screening_start = time.perf_counter()

    def run_screen_candidate(configuration, group):
        context = {"stage": "development", "model": configuration["model_name"], "fold": 0,
                   "sensitive_mode": "without_sensitive", "target_mode": configuration["target_mode"]}
        try:
            fitted, fit_seconds, retry_number, convergence_status = fit_with_retries(
                configuration, X_dev_train, y_dev_train, numeric_without, categorical_without, context
            )
        except RuntimeError as error:
            screening_rows.append({
                "screening_group": group, "candidate_id": candidate_key(configuration, group),
                "model_name": configuration["model_name"], "target_mode": configuration["target_mode"],
                "scaler": configuration["scaler"], "alpha": configuration.get("alpha"),
                "l1_ratio": configuration.get("l1_ratio"), "parameter_json": canonical_json(configuration),
                **{key: None for key in ["mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero",
                                           "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error",
                                           "negative_prediction_rate", "mae_usd", "rmse_usd"]},
                "transformed_features": None, "nonzero_coefficients": None,
                "fit_time_seconds": None, "prediction_time_seconds": None,
                "retry_number": 2, "convergence_status": "failed_after_retries", "status": "failed",
            })
            experiment_id = deterministic_experiment_id(
                configuration["model_name"], "without_sensitive", configuration["target_mode"],
                f"development_{group}", 0, configuration
            )
            prompt2_registry_records.append(registry_record(
                experiment_id, configuration, "without_sensitive", f"development_{group}", 0,
                len(y_dev_train), len(y_dev_validation), None, None, None,
                status="failed", notes=str(error)
            ))
            gc.collect()
            return
        prediction, prediction_seconds = predict_and_measure(fitted, X_dev_validation)
        metrics = evaluate_regression_predictions(y_dev_validation, prediction)
        diagnostics = model_diagnostics(fitted)
        row = {
            "screening_group": group, "candidate_id": candidate_key(configuration, group),
            "model_name": configuration["model_name"], "target_mode": configuration["target_mode"],
            "scaler": configuration["scaler"], "alpha": configuration.get("alpha"),
            "l1_ratio": configuration.get("l1_ratio"), "parameter_json": canonical_json(configuration),
            **{key: value for key, value in metrics.items() if key != "metric_warnings"},
            **diagnostics, "fit_time_seconds": fit_seconds, "prediction_time_seconds": prediction_seconds,
            "retry_number": retry_number, "convergence_status": convergence_status, "status": "success",
        }
        screening_rows.append(row)
        experiment_id = deterministic_experiment_id(
            configuration["model_name"], "without_sensitive", configuration["target_mode"],
            f"development_{group}", 0, configuration
        )
        prompt2_registry_records.append(registry_record(
            experiment_id, configuration, "without_sensitive", f"development_{group}", 0,
            len(y_dev_train), len(y_dev_validation), metrics, fit_seconds, prediction_seconds,
            notes=f"Development screening; retry={retry_number}; convergence={convergence_status}"
        ))
        del fitted, prediction
        gc.collect()

    for configuration in scaler_candidates:
        run_screen_candidate(configuration, "scaler")
    scaler_frame = pd.DataFrame(screening_rows)
    best_scaler_mae = scaler_frame["mae"].min()
    equivalent_scalers = scaler_frame.loc[scaler_frame["mae"] < best_scaler_mae * 1.005].copy()
    equivalent_scalers["standard_preference"] = (equivalent_scalers["scaler"] == "standard").astype(int)
    selected_scaler = equivalent_scalers.sort_values(
        ["standard_preference", "fit_time_seconds", "candidate_id"], ascending=[False, True, True]
    ).iloc[0]["scaler"]

    candidate_grid = model_candidate_grid(selected_scaler)
    for family, candidates in candidate_grid.items():
        for configuration in candidates:
            run_screen_candidate(configuration, family)

    development_screening_results = pd.DataFrame(screening_rows)

    def select_model_configuration(family):
        frame = development_screening_results.loc[
            (development_screening_results["screening_group"] == family)
            & (development_screening_results["status"] == "success")
        ].copy()
        if frame.empty:
            raise RuntimeError(f"No successful development configuration remains for required family {family}.")
        best = frame["mae"].min()
        eligible = frame.loc[frame["mae"] < best * 1.0025].copy()
        eligible["raw_preference"] = (eligible["target_mode"] == "raw").astype(int)
        eligible["alpha_rank"] = eligible["alpha"].fillna(-np.inf)
        eligible["nonzero_rank"] = eligible["nonzero_coefficients"].fillna(np.inf)
        eligible["runtime"] = eligible["fit_time_seconds"] + eligible["prediction_time_seconds"]
        selected = eligible.sort_values(
            ["raw_preference", "alpha_rank", "nonzero_rank", "runtime", "candidate_id"],
            ascending=[False, False, True, True, True]
        ).iloc[0]
        return json.loads(selected["parameter_json"]), selected["candidate_id"], float(best), len(eligible)

    selected_models = {}
    selection_evidence = {}
    for family in ["dummy_median", "linear_regression", "ridge", "lasso", "elastic_net", "gamma_regressor"]:
        configuration, selected_candidate_id, best_mae, equivalent_count = select_model_configuration(family)
        selected_models[family] = configuration
        selection_evidence[family] = {"selected_candidate_id": selected_candidate_id, "best_development_mae": best_mae,
                                      "equivalent_candidate_count": equivalent_count,
                                      "tie_rule": "raw, stronger alpha, fewer nonzero, runtime, stable ID"}
    selected_linear_configurations = {
        "version": PROMPT2_VERSION, "selected_scaler": selected_scaler,
        "development_manifest_sha256": development_manifest_hash,
        "selection_primary_metric": "mae", "models": selected_models,
        "selection_evidence": selection_evidence,
    }
    development_screening_results.to_csv(screening_results_path, index=False)
    selected_configurations_path.write_text(json.dumps(selected_linear_configurations, indent=2), encoding="utf-8")
    screening_runtime_seconds = time.perf_counter() - screening_start
else:
    selected_scaler = selected_linear_configurations["selected_scaler"]
    screening_runtime_seconds = 0.0
    for row in development_screening_results.itertuples():
        configuration = json.loads(row.parameter_json)
        experiment_id = deterministic_experiment_id(
            configuration["model_name"], "without_sensitive", configuration["target_mode"],
            f"development_{row.screening_group}", 0, configuration
        )
        metrics = {key: getattr(row, key) for key in ["mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero",
                                                            "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error", "negative_prediction_rate"]}
        prompt2_registry_records.append(registry_record(
            experiment_id, configuration, "without_sensitive", f"development_{row.screening_group}", 0,
            len(y_dev_train), len(y_dev_validation), metrics, row.fit_time_seconds, row.prediction_time_seconds,
            status=row.status, notes=f"Validated cached development screening; convergence={row.convergence_status}"
        ))
print({"screening_candidates": len(development_screening_results), "selected_scaler": selected_scaler,
       "cache_reused": screening_cache_valid, "screening_runtime_seconds": round(screening_runtime_seconds, 2)})
display(development_screening_results.loc[development_screening_results.screening_group == "scaler",
        ["scaler", "target_mode", "mae", "rmse", "rmsle", "fit_time_seconds"]].sort_values("mae"))

{'screening_candidates': 47, 'selected_scaler': 'standard', 'cache_reused': np.True_, 'screening_runtime_seconds': 0.0}


,scaler,target_mode,mae,rmse,rmsle,fit_time_seconds
1,standard,log1p,70.663967,132.509667,0.438922,1.339546
3,robust,log1p,70.667964,132.479845,0.438877,1.351113
0,standard,raw,86.158237,154.929329,NaN,1.311156
2,robust,raw,86.165095,154.961851,NaN,1.274837


Scaler choice uses development MAE only. If candidates are within 0.5 percent, Standard scaling has the deterministic simplicity preference. The printed result shows the real selected policy.

## 24. Dummy Regressor

The median dummy gives a simple reference. It uses raw target values and the full preprocessing policy.

In [25]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "dummy_median",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["dummy_median"]
selected_id = selected_linear_configurations["selection_evidence"]["dummy_median"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `dummy_median` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
4,raw,NaN,NaN,131.86075,232.954527,0.922863,0.0,NaN,0.939189,0,converged,dummy_median__8b0d8987c0c1


**Selected result:** `dummy_median` uses `raw` target with development MAE 131.861 target units. Parameters: `{"model_name":"dummy_median","scaler":"standard","target_mode":"raw"}`. Convergence status: converged.

## 25. Linear Regression

Linear Regression estimates one additive coefficient for each transformed feature. Raw and log targets are compared.

In [26]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "linear_regression",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["linear_regression"]
selected_id = selected_linear_configurations["selection_evidence"]["linear_regression"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `linear_regression` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
6,log1p,NaN,NaN,70.666676,132.501414,0.438961,0.00000,97.0,2.437255,0,converged,linear_regression__7a4d325a3652
5,raw,NaN,NaN,86.216167,154.975006,NaN,0.03605,97.0,2.079093,0,converged,linear_regression__28961a11ade0


**Selected result:** `linear_regression` uses `log1p` target with development MAE 70.667 target units. Parameters: `{"model_name":"linear_regression","scaler":"standard","target_mode":"log1p"}`. Convergence status: converged.

## 26. Ridge Regression

Ridge shrinks coefficients with an L2 penalty. It can be more stable when features are correlated.

In [27]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "ridge",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["ridge"]
selected_id = selected_linear_configurations["selection_evidence"]["ridge"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `ridge` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
12,log1p,0.01,NaN,70.663159,132.506891,0.438920,0.00000,97.0,1.431492,0,converged,ridge__b342a02b5cbc
13,log1p,0.10,NaN,70.663233,132.507142,0.438920,0.00000,97.0,1.427045,0,converged,ridge__80cea3e2b44e
14,log1p,1.00,NaN,70.663967,132.509667,0.438922,0.00000,97.0,1.421099,0,converged,ridge__89d6ca22b7e7
15,log1p,10.00,NaN,70.671918,132.535740,0.438941,0.00000,97.0,1.386684,0,converged,ridge__53afa6ee73ca
16,log1p,100.00,NaN,70.782118,132.833500,0.439416,0.00000,97.0,1.351282,0,converged,ridge__091fc2b9df3a
11,raw,100.00,NaN,85.767669,154.536861,NaN,0.03500,97.0,1.294899,0,converged,ridge__46f2a1ef16cd
10,raw,10.00,NaN,86.116057,154.885618,NaN,0.03600,97.0,1.337376,0,converged,ridge__fc9e4c2302ba
9,raw,1.00,NaN,86.158237,154.929329,NaN,0.03605,97.0,1.249910,0,converged,ridge__5236ff00c4a7


**Selected result:** `ridge` uses `log1p` target with development MAE 70.782 target units. Parameters: `{"alpha":100.0,"model_name":"ridge","scaler":"standard","target_mode":"log1p"}`. Convergence status: converged.

## 27. Lasso Regression

Lasso uses an L1 penalty and can set coefficients to zero. Convergence warnings are captured and retried.

In [28]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "lasso",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["lasso"]
selected_id = selected_linear_configurations["selection_evidence"]["lasso"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `lasso` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
21,log1p,0.00001,NaN,70.665902,132.503791,0.438948,0.00000,87.0,82.209421,1,converged,lasso__db5fa4f5e1dc
22,log1p,0.00010,NaN,70.694198,132.545418,0.439072,0.00000,77.0,4.807466,0,converged,lasso__6deeea34e8f3
23,log1p,0.00100,NaN,71.294682,133.386111,0.441838,0.00000,51.0,3.906520,0,converged,lasso__cac97737458e
24,log1p,0.01000,NaN,75.235890,138.738516,0.465975,0.00000,21.0,1.686808,0,converged,lasso__bff42cbe24c1
19,raw,1.00000,NaN,85.613361,155.622401,NaN,0.02870,32.0,4.292757,0,converged,lasso__c10a54bacf81
18,raw,0.10000,NaN,86.015525,154.998039,NaN,0.03520,65.0,5.211044,0,converged,lasso__4c82aa6eff14
17,raw,0.01000,NaN,86.162149,154.960069,NaN,0.03605,85.0,7.262550,0,converged,lasso__4107599d7c43
20,raw,10.00000,NaN,96.646695,169.853390,NaN,0.00470,9.0,2.228840,0,converged,lasso__146f459f8b49


**Selected result:** `lasso` uses `log1p` target with development MAE 70.694 target units. Parameters: `{"alpha":0.0001,"max_iter":5000,"model_name":"lasso","scaler":"standard","target_mode":"log1p","tol":0.0001}`. Convergence status: converged.

## 28. ElasticNet Regression

ElasticNet mixes L1 and L2 penalties. It can combine shrinkage with sparse coefficients.

In [29]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "elastic_net",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["elastic_net"]
selected_id = selected_linear_configurations["selection_evidence"]["elastic_net"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `elastic_net` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
35,log1p,0.00001,0.5,70.665446,132.503889,0.438948,0.0,92.0,336.061864,2,converged,elastic_net__f43afbc1fdd1
36,log1p,0.00001,0.8,70.665477,132.503563,0.438946,0.0,88.0,128.862764,1,converged,elastic_net__b265dc5cdce0
37,log1p,0.00010,0.2,70.673228,132.529160,0.438955,0.0,89.0,26.388170,0,converged,elastic_net__59c885c5e749
38,log1p,0.00010,0.5,70.680049,132.533436,0.438995,0.0,84.0,16.011154,0,converged,elastic_net__7f9d9737f95e
39,log1p,0.00010,0.8,70.687534,132.539590,0.439037,0.0,79.0,5.171965,0,converged,elastic_net__0d470700df89
40,log1p,0.00100,0.2,70.833003,132.833343,0.439670,0.0,73.0,2.969569,0,converged,elastic_net__3361801d1033
41,log1p,0.00100,0.5,70.977843,132.991679,0.440316,0.0,64.0,2.987566,0,converged,elastic_net__1d59f943e6a0
42,log1p,0.00100,0.8,71.177439,133.303348,0.441184,0.0,56.0,2.765104,0,converged,elastic_net__97ad852ce863


**Selected result:** `elastic_net` uses `log1p` target with development MAE 70.833 target units. Parameters: `{"alpha":0.001,"l1_ratio":0.2,"max_iter":5000,"model_name":"elastic_net","scaler":"standard","target_mode":"log1p","tol":0.0001}`. Convergence status: converged.

## 29. Gamma Regression

Gamma regression models a positive mean with a log link. It uses the raw positive target and tests regularization strength.

In [30]:
family_results = development_screening_results.loc[
    development_screening_results["screening_group"] == "gamma_regressor",
    ["target_mode", "alpha", "l1_ratio", "mae", "rmse", "rmsle", "negative_prediction_rate",
     "nonzero_coefficients", "fit_time_seconds", "retry_number", "convergence_status", "candidate_id"]
].sort_values("mae")
selected_family_config = selected_linear_configurations["models"]["gamma_regressor"]
selected_id = selected_linear_configurations["selection_evidence"]["gamma_regressor"]["selected_candidate_id"]
selected_row = family_results.loc[family_results["candidate_id"] == selected_id].iloc[0]
display(family_results.head(8))
display(Markdown(
    f"**Selected result:** `gamma_regressor` uses `{selected_family_config['target_mode']}` target with "
    f"development MAE {selected_row['mae']:.3f} target units. Parameters: `{canonical_json(selected_family_config)}`. "
    f"Convergence status: {selected_row['convergence_status']}."
))

,target_mode,alpha,l1_ratio,mae,rmse,rmsle,negative_prediction_rate,nonzero_coefficients,fit_time_seconds,retry_number,convergence_status,candidate_id
43,raw,0.00,NaN,71.177423,129.874866,0.454239,0.0,97.0,5.092972,0,converged,gamma_regressor__3ed9ebb6e05d
44,raw,0.01,NaN,71.669850,131.087796,0.467518,0.0,97.0,2.025682,0,converged,gamma_regressor__53d9fec5147e
45,raw,0.10,NaN,77.089691,141.219707,0.582562,0.0,97.0,1.401547,0,converged,gamma_regressor__c361a0055150
46,raw,1.00,NaN,95.283352,169.970364,0.761998,0.0,97.0,1.133856,0,converged,gamma_regressor__b59b29eb13e3


**Selected result:** `gamma_regressor` uses `raw` target with development MAE 71.177 target units. Parameters: `{"alpha":0.0,"max_iter":500,"model_name":"gamma_regressor","scaler":"standard","target_mode":"raw","tol":1e-05}`. Convergence status: converged.

## 30. Frozen Model Configurations

Each family now has one frozen configuration selected from non-sensitive development data. The exact same configuration will be used for both sensitive modes.

In [31]:
selected_models = selected_linear_configurations["models"]
required_model_families = ["dummy_median", "linear_regression", "ridge", "lasso", "elastic_net", "gamma_regressor"]
if list(selected_models) != required_model_families:
    raise AssertionError("Selected model family order or membership is invalid.")
controlled_configuration_hashes = {
    family: {
        "without_sensitive": configuration_digest(selected_models[family]),
        "with_sensitive": configuration_digest(selected_models[family]),
    } for family in required_model_families
}
if not all(v["without_sensitive"] == v["with_sensitive"] for v in controlled_configuration_hashes.values()):
    raise AssertionError("Sensitive modes do not share frozen configurations.")
frozen_config_table = pd.DataFrame([
    {"model": family, "target_mode": config["target_mode"], "scaler": config["scaler"],
     "alpha": config.get("alpha"), "l1_ratio": config.get("l1_ratio"),
     "configuration_hash": configuration_digest(config)}
    for family, config in selected_models.items()
])
display(frozen_config_table)

,model,target_mode,scaler,alpha,l1_ratio,configuration_hash
0,dummy_median,raw,standard,NaN,NaN,8b0d8987c0c1
1,linear_regression,log1p,standard,NaN,NaN,7a4d325a3652
2,ridge,log1p,standard,100.0000,NaN,091fc2b9df3a
3,lasso,log1p,standard,0.0001,NaN,6deeea34e8f3
4,elastic_net,log1p,standard,0.0010,0.2,3361801d1033
5,gamma_regressor,raw,standard,0.0000,NaN,3ed9ebb6e05d


The six configurations are frozen. Sensitive features cannot influence target-mode, scaler, alpha, or L1-ratio selection.

## 31. Full Three-Fold Cross-Validation

Each model and mode uses three fresh complete pipelines and the saved folds. Every training row receives one OOF prediction. Fits run sequentially.

In [32]:
cv_fold_results_path = P2_DIRS["results"] / "cv_fold_results.csv"
cv_oof_summary_path = P2_DIRS["results"] / "cv_oof_summary.csv"
warnings_path = P2_DIRS["results"] / "model_warnings.csv"
cv_manifest_path = P2_DIRS["manifests"] / "prompt2_cv_manifest.json"
selected_configuration_digest = configuration_digest(selected_linear_configurations)
expected_oof_paths = {
    (family, mode): P2_DIRS["predictions"] / f"{family}__{mode}__oof.csv"
    for family in required_model_families for mode in ["without_sensitive", "with_sensitive"]
}
cv_cache_valid = False
if cv_fold_results_path.exists() and cv_oof_summary_path.exists() and cv_manifest_path.exists() and all(path.exists() for path in expected_oof_paths.values()):
    cached_manifest = json.loads(cv_manifest_path.read_text(encoding="utf-8"))
    cv_fold_results = pd.read_csv(cv_fold_results_path)
    cv_oof_summary = pd.read_csv(cv_oof_summary_path)
    cv_cache_valid = (
        cached_manifest.get("selected_configuration_digest") == selected_configuration_digest
        and cached_manifest.get("development_manifest_sha256") == development_manifest_hash
        and len(cv_fold_results) == 36 and len(cv_oof_summary) == 12
        and (cv_fold_results["status"] == "success").all()
    )
    if cv_cache_valid:
        for (family, mode), path in expected_oof_paths.items():
            check_frame = pd.read_csv(path, usecols=["row_id", "fold", "y_pred"])
            cv_cache_valid = cv_cache_valid and (
                len(check_frame) == len(train_ids)
                and not check_frame["row_id"].duplicated().any()
                and np.array_equal(np.sort(check_frame["row_id"].to_numpy()), train_ids)
                and len(set(check_frame["row_id"]) & test_id_set) == 0
                and np.isfinite(check_frame["y_pred"]).all()
            )

if not cv_cache_valid:
    cv_start = time.perf_counter()
    fold_rows = []
    oof_summary_rows = []
    for family in required_model_families:
        configuration = selected_models[family]
        for sensitive_mode in ["without_sensitive", "with_sensitive"]:
            if sensitive_mode == "without_sensitive":
                X_mode, numeric_mode, categorical_mode = X_without_train, numeric_without, categorical_without
            else:
                X_mode, numeric_mode, categorical_mode = X_with_train, numeric_with, categorical_with
            oof_prediction = np.full(len(train_ids), np.nan, dtype=float)
            fold_written = np.zeros(len(train_ids), dtype=bool)
            for fold in [0, 1, 2]:
                validation_positions = np.flatnonzero(fold_by_train_position == fold)
                training_positions = np.flatnonzero(fold_by_train_position != fold)
                context = {"stage": "cv", "model": family, "fold": fold, "sensitive_mode": sensitive_mode,
                           "target_mode": configuration["target_mode"]}
                fitted, fit_seconds, retry_number, convergence_status = fit_with_retries(
                    configuration, X_mode.iloc[training_positions], y_train_prompt2[training_positions],
                    numeric_mode, categorical_mode, context
                )
                prediction, prediction_seconds = predict_and_measure(fitted, X_mode.iloc[validation_positions])
                if fold_written[validation_positions].any():
                    raise AssertionError("An OOF position was written more than once.")
                oof_prediction[validation_positions] = prediction
                fold_written[validation_positions] = True
                metrics = evaluate_regression_predictions(y_train_prompt2[validation_positions], prediction)
                diagnostics = model_diagnostics(fitted)
                fold_experiment_id = deterministic_experiment_id(
                    family, sensitive_mode, configuration["target_mode"], "cv_fold", fold, configuration
                )
                fold_row = {
                    "experiment_id": fold_experiment_id, "model_name": family, "sensitive_mode": sensitive_mode,
                    "target_mode": configuration["target_mode"], "fold": fold,
                    **{key: value for key, value in metrics.items() if key != "metric_warnings"},
                    **diagnostics, "fit_time_seconds": fit_seconds, "prediction_time_seconds": prediction_seconds,
                    "retry_number": retry_number, "convergence_status": convergence_status, "status": "success",
                    "parameter_json": canonical_json(configuration),
                }
                fold_rows.append(fold_row)
                prompt2_registry_records.append(registry_record(
                    fold_experiment_id, configuration, sensitive_mode, "cv_fold", fold,
                    len(training_positions), len(validation_positions), metrics, fit_seconds, prediction_seconds,
                    notes=f"Fresh saved-fold pipeline; retry={retry_number}; convergence={convergence_status}"
                ))
                del fitted, prediction
                gc.collect()
            if not fold_written.all() or not np.isfinite(oof_prediction).all():
                raise AssertionError(f"OOF coverage failed for {family}, {sensitive_mode}.")
            oof_experiment_id = deterministic_experiment_id(
                family, sensitive_mode, configuration["target_mode"], "oof_summary", None, configuration
            )
            oof_path = expected_oof_paths[(family, sensitive_mode)]
            oof_frame = pd.DataFrame({
                "row_id": train_ids, "fold": fold_by_train_position, "y_true": y_train_prompt2,
                "y_pred": oof_prediction, "absolute_error": np.abs(oof_prediction - y_train_prompt2),
                "signed_error": oof_prediction - y_train_prompt2, "model_name": family,
                "sensitive_mode": sensitive_mode, "target_mode": configuration["target_mode"],
                "feature_set": PROMPT2_VERSION, "experiment_id": oof_experiment_id,
            })
            oof_frame.to_csv(oof_path, index=False)
            oof_metrics = evaluate_regression_predictions(y_train_prompt2, oof_prediction)
            family_folds = [row for row in fold_rows if row["model_name"] == family and row["sensitive_mode"] == sensitive_mode]
            summary_row = {
                "experiment_id": oof_experiment_id, "model_name": family, "sensitive_mode": sensitive_mode,
                "target_mode": configuration["target_mode"], "parameter_json": canonical_json(configuration),
                **{key: value for key, value in oof_metrics.items() if key != "metric_warnings"},
                "fold_mae_mean": float(np.mean([row["mae"] for row in family_folds])),
                "fold_mae_std": float(np.std([row["mae"] for row in family_folds], ddof=1)),
                "total_fit_time_seconds": float(sum(row["fit_time_seconds"] for row in family_folds)),
                "total_prediction_time_seconds": float(sum(row["prediction_time_seconds"] for row in family_folds)),
                "transformed_features": int(max(row["transformed_features"] for row in family_folds)),
                "nonzero_coefficients": family_folds[-1]["nonzero_coefficients"],
                "convergence_status": "converged" if all(row["convergence_status"] == "converged" for row in family_folds) else "warning",
                "oof_path": str(oof_path.relative_to(PROJECT_ROOT)),
            }
            oof_summary_rows.append(summary_row)
            prompt2_registry_records.append(registry_record(
                oof_experiment_id, configuration, sensitive_mode, "oof_summary", None,
                len(train_ids), len(train_ids), oof_metrics, summary_row["total_fit_time_seconds"],
                summary_row["total_prediction_time_seconds"], notes="Metrics from the complete OOF vector",
                prediction_path=str(oof_path.relative_to(PROJECT_ROOT))
            ))
            del oof_frame, oof_prediction
            gc.collect()
    cv_fold_results = pd.DataFrame(fold_rows)
    cv_oof_summary = pd.DataFrame(oof_summary_rows)
    cv_fold_results.to_csv(cv_fold_results_path, index=False)
    cv_oof_summary.to_csv(cv_oof_summary_path, index=False)
    cv_runtime_seconds = time.perf_counter() - cv_start
    cv_manifest = {
        "version": PROMPT2_VERSION, "selected_configuration_digest": selected_configuration_digest,
        "development_manifest_sha256": development_manifest_hash, "fold_evaluation_count": len(cv_fold_results),
        "oof_file_count": len(expected_oof_paths), "fresh_pipeline_fit_count": len(cv_fold_results),
        "cv_runtime_seconds": cv_runtime_seconds,
    }
    cv_manifest_path.write_text(json.dumps(cv_manifest, indent=2), encoding="utf-8")
else:
    cv_runtime_seconds = 0.0
    for row in cv_fold_results.itertuples():
        configuration = json.loads(row.parameter_json)
        metrics = {key: getattr(row, key) for key in ["mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero",
                                                            "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error", "negative_prediction_rate"]}
        prompt2_registry_records.append(registry_record(
            row.experiment_id, configuration, row.sensitive_mode, "cv_fold", int(row.fold),
            int((fold_by_train_position != row.fold).sum()), int((fold_by_train_position == row.fold).sum()),
            metrics, row.fit_time_seconds, row.prediction_time_seconds, notes="Validated cached saved-fold result"
        ))
    for row in cv_oof_summary.itertuples():
        configuration = json.loads(row.parameter_json)
        metrics = {key: getattr(row, key) for key in ["mae", "mse", "rmse", "mape_percent", "r_squared", "rmsle", "rmsle_clipped_zero",
                                                            "median_absolute_error", "wape_percent", "mean_signed_error", "p90_absolute_error", "negative_prediction_rate"]}
        prompt2_registry_records.append(registry_record(
            row.experiment_id, configuration, row.sensitive_mode, "oof_summary", None,
            len(train_ids), len(train_ids), metrics, row.total_fit_time_seconds, row.total_prediction_time_seconds,
            notes="Validated cached complete OOF vector", prediction_path=row.oof_path
        ))
print({"fold_evaluations": len(cv_fold_results), "oof_summaries": len(cv_oof_summary),
       "cache_reused": cv_cache_valid, "cv_runtime_seconds": round(cv_runtime_seconds, 2)})
display(cv_fold_results.groupby(["model_name", "sensitive_mode"])["mae"].agg(["mean", "std"]).head(12))

{'fold_evaluations': 36, 'oof_summaries': 12, 'cache_reused': np.True_, 'cv_runtime_seconds': 0.0}


mean       std
model_name        sensitive_mode                         
dummy_median      with_sensitive     132.688445  0.434532
                  without_sensitive  132.688445  0.434532
elastic_net       with_sensitive      72.273614  0.288793
                  without_sensitive   72.369725  0.308580
...                                         ...       ...
linear_regression with_sensitive      72.312180  0.245321
                  without_sensitive   72.409824  0.224541
ridge             with_sensitive      72.215772  0.188016
                  without_sensitive   72.311402  0.203481

[12 rows x 2 columns]

All 36 saved-fold evaluations and 12 complete OOF vectors are present. Aggregate metrics are calculated from each full OOF vector, not from averaged fold metrics.

## 32. OOF Evaluation Results

The leaderboard compares every required model and both sensitive modes on original-scale training OOF predictions.

In [33]:
linear_leaderboard = cv_oof_summary.rename(columns={
    "mae": "oof_mae", "rmse": "oof_rmse", "rmsle": "oof_rmsle", "r_squared": "oof_r_squared"
}).copy()
linear_leaderboard = linear_leaderboard.sort_values("oof_mae").reset_index(drop=True)
leaderboard_path = P2_DIRS["results"] / "linear_leaderboard.csv"
linear_leaderboard.to_csv(leaderboard_path, index=False)
best_linear_row = linear_leaderboard.iloc[0]
display(linear_leaderboard[["model_name", "sensitive_mode", "target_mode", "oof_mae", "fold_mae_mean", "fold_mae_std",
                            "oof_rmse", "oof_rmsle", "oof_r_squared", "negative_prediction_rate", "convergence_status"]])
display(Markdown(
    f"**Current training-OOF leader:** `{best_linear_row['model_name']}` in `{best_linear_row['sensitive_mode']}` mode "
    f"has MAE {best_linear_row['oof_mae']:.3f} target units. This is only the best current linear-family result, not the final project model."
))

,model_name,sensitive_mode,target_mode,oof_mae,fold_mae_mean,fold_mae_std,oof_rmse,oof_rmsle,oof_r_squared,negative_prediction_rate,convergence_status
0,lasso,with_sensitive,log1p,72.126971,72.126971,0.270256,2.480037e+02,0.440883,3.907267e-01,0.0,converged
1,ridge,with_sensitive,log1p,72.215772,72.215772,0.188016,2.609834e+02,0.440819,3.252830e-01,0.0,converged
2,lasso,without_sensitive,log1p,72.230441,72.230441,0.280963,2.452373e+02,0.441602,4.042432e-01,0.0,converged
3,elastic_net,with_sensitive,log1p,72.273615,72.273614,0.288793,2.507411e+02,0.441413,3.772024e-01,0.0,converged
...,...,...,...,...,...,...,...,...,...,...,...
8,dummy_median,with_sensitive,raw,132.688445,132.688445,0.434532,3.216897e+02,0.921838,-2.510925e-02,0.0,converged
9,dummy_median,without_sensitive,raw,132.688445,132.688445,0.434532,3.216897e+02,0.921838,-2.510925e-02,0.0,converged
10,gamma_regressor,with_sensitive,raw,6465.449302,6465.433364,11054.399874,4.027393e+06,0.455561,-1.606733e+08,0.0,converged
11,gamma_regressor,without_sensitive,raw,9380.022139,9379.998910,16102.456916,5.868624e+06,0.456429,-3.411681e+08,0.0,converged


**Current training-OOF leader:** `lasso` in `with_sensitive` mode has MAE 72.127 target units. This is only the best current linear-family result, not the final project model.

## 33. Sensitive Feature Comparison

This paired table defines each difference as `with_sensitive - without_sensitive`. A positive MAE difference means the sensitive version was worse.

In [34]:
comparison_rows = []
for family in required_model_families:
    without = cv_oof_summary.loc[(cv_oof_summary.model_name == family) & (cv_oof_summary.sensitive_mode == "without_sensitive")].iloc[0]
    with_sensitive = cv_oof_summary.loc[(cv_oof_summary.model_name == family) & (cv_oof_summary.sensitive_mode == "with_sensitive")].iloc[0]
    comparison_rows.append({
        "model_name": family, "mae_without_sensitive": without.mae, "mae_with_sensitive": with_sensitive.mae,
        "mae_difference_with_minus_without": with_sensitive.mae - without.mae,
        "relative_mae_difference_percent": (with_sensitive.mae - without.mae) / without.mae * 100,
        "rmse_difference_with_minus_without": with_sensitive.rmse - without.rmse,
        "rmsle_difference_with_minus_without": with_sensitive.rmsle - without.rmsle if pd.notna(with_sensitive.rmsle) and pd.notna(without.rmsle) else np.nan,
        "r_squared_difference_with_minus_without": with_sensitive.r_squared - without.r_squared,
        "runtime_difference_seconds": (with_sensitive.total_fit_time_seconds + with_sensitive.total_prediction_time_seconds)
                                      - (without.total_fit_time_seconds + without.total_prediction_time_seconds),
    })
linear_sensitive_comparison = pd.DataFrame(comparison_rows)
linear_sensitive_comparison.to_csv(P2_DIRS["results"] / "linear_sensitive_comparison.csv", index=False)
display(linear_sensitive_comparison)
display(Markdown(
    "Sensitive fields may improve, worsen, or not change predictive accuracy. This controlled accuracy comparison is not a fairness audit. "
    "The non-sensitive pack can still contain proxy variables."
))

,model_name,mae_without_sensitive,mae_with_sensitive,mae_difference_with_minus_without,relative_mae_difference_percent,rmse_difference_with_minus_without,rmsle_difference_with_minus_without,r_squared_difference_with_minus_without,runtime_difference_seconds
0,dummy_median,132.688445,132.688445,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,4.993444
1,linear_regression,72.409824,72.312179,-0.097645,-0.134850,1.179809e+00,-0.000748,-6.905915e-03,18.585258
2,ridge,72.311403,72.215772,-0.095631,-0.132248,3.145933e+00,-0.000731,-1.616824e-02,7.336428
3,lasso,72.230441,72.126971,-0.103470,-0.143250,2.766369e+00,-0.000719,-1.351653e-02,20.036070
4,elastic_net,72.369726,72.273615,-0.096111,-0.132806,4.494408e+00,-0.000707,-2.212657e-02,13.073822
5,gamma_regressor,9380.022139,6465.449302,-2914.572836,-31.072132,-1.841231e+06,-0.000868,1.804947e+08,85.020314


Sensitive fields may improve, worsen, or not change predictive accuracy. This controlled accuracy comparison is not a fairness audit. The non-sensitive pack can still contain proxy variables.

## 34. Coefficient and Sparsity Review

Coefficients will be extracted from final fitted pipelines with safe transformed feature names. They describe model associations, not causes. Scaling, log targets, the Gamma log link, and correlated fields change their meaning.

In [35]:
def coefficient_frame(model, family, sensitive_mode, target_mode):
    estimator = fitted_estimator(model)
    coefficients = getattr(estimator, "coef_", None)
    if coefficients is None:
        return None
    coefficients = np.ravel(np.asarray(coefficients, dtype=float))
    names = transformed_feature_names(model)
    if len(names) != len(coefficients):
        raise AssertionError("Coefficient count does not match transformed feature names.")
    return pd.DataFrame({
        "transformed_feature": names, "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients), "model_name": family,
        "sensitive_mode": sensitive_mode, "target_mode": target_mode,
        "nonzero": coefficients != 0,
    })
print("Coefficient extraction utility is ready. Final models are fitted in the next section.")

Coefficient extraction utility is ready. Final models are fitted in the next section.


## 35. Final Training Pipeline Fits

One fresh complete pipeline per model and mode is fitted on all saved training rows. No test row is included. The full preprocessing and target transformation are saved inside each joblib object.

In [36]:
model_manifest_path = P2_DIRS["manifests"] / "prompt2_model_manifest.json"
coefficient_diagnostics_path = P2_DIRS["features"] / "coefficient_diagnostics.csv"
reload_reference_path = P2_DIRS["manifests"] / "prompt2_reload_reference.csv"
expected_model_keys = [(family, mode) for family in required_model_families for mode in ["without_sensitive", "with_sensitive"]]
expected_model_paths = {
    key: P2_DIRS["models"] / f"{key[0]}__{key[1]}__{selected_models[key[0]]['target_mode']}.joblib"
    for key in expected_model_keys
}
final_cache_valid = model_manifest_path.exists() and all(path.exists() for path in expected_model_paths.values())
if final_cache_valid:
    model_manifest = json.loads(model_manifest_path.read_text(encoding="utf-8"))
    final_cache_valid = (
        model_manifest.get("selected_configuration_digest") == selected_configuration_digest
        and len(model_manifest.get("models", [])) == 12
    )

final_models = {}
final_fit_rows = []
coefficient_diagnostics = []
reload_reference_rows = []
reload_sample_positions = np.arange(0, min(20, len(train_ids)), dtype=int)
if not final_cache_valid:
    final_start = time.perf_counter()
    model_manifest_rows = []
    for family, sensitive_mode in expected_model_keys:
        configuration = selected_models[family]
        if sensitive_mode == "without_sensitive":
            X_mode, numeric_mode, categorical_mode = X_without_train, numeric_without, categorical_without
        else:
            X_mode, numeric_mode, categorical_mode = X_with_train, numeric_with, categorical_with
        context = {"stage": "final_train", "model": family, "fold": None, "sensitive_mode": sensitive_mode,
                   "target_mode": configuration["target_mode"]}
        fitted, fit_seconds, retry_number, convergence_status = fit_with_retries(
            configuration, X_mode, y_train_prompt2, numeric_mode, categorical_mode, context
        )
        prediction, prediction_seconds = predict_and_measure(fitted, X_mode.iloc[reload_sample_positions])
        model_path = expected_model_paths[(family, sensitive_mode)]
        joblib.dump(fitted, model_path, compress=3)
        model_sha256 = file_sha256(model_path)
        diagnostics = model_diagnostics(fitted)
        final_experiment_id = deterministic_experiment_id(
            family, sensitive_mode, configuration["target_mode"], "final_train", None, configuration
        )
        final_fit_rows.append({
            "experiment_id": final_experiment_id, "model_name": family, "sensitive_mode": sensitive_mode,
            "target_mode": configuration["target_mode"], "fit_time_seconds": fit_seconds,
            "prediction_time_seconds": prediction_seconds, **diagnostics,
            "retry_number": retry_number, "convergence_status": convergence_status,
            "model_path": str(model_path.relative_to(PROJECT_ROOT)), "model_sha256": model_sha256,
            "parameter_json": canonical_json(configuration),
        })
        prompt2_registry_records.append(registry_record(
            final_experiment_id, configuration, sensitive_mode, "final_train", None,
            len(train_ids), 0, None, fit_seconds, prediction_seconds,
            notes=f"Full training fit; retry={retry_number}; convergence={convergence_status}",
            model_path=str(model_path.relative_to(PROJECT_ROOT))
        ))
        for row_id_value, prediction_value in zip(train_ids[reload_sample_positions], prediction):
            reload_reference_rows.append({"model_name": family, "sensitive_mode": sensitive_mode,
                                          "row_id": int(row_id_value), "reference_prediction": float(prediction_value)})
        coefficients = coefficient_frame(fitted, family, sensitive_mode, configuration["target_mode"])
        if coefficients is not None:
            coefficient_path = P2_DIRS["coefficients"] / f"{family}__{sensitive_mode}__coefficients.csv"
            coefficients.to_csv(coefficient_path, index=False)
            coefficient_diagnostics.append({
                "model_name": family, "sensitive_mode": sensitive_mode, "target_mode": configuration["target_mode"],
                "total_transformed_features": len(coefficients), "nonzero_coefficients": int(coefficients["nonzero"].sum()),
                "zero_coefficients": int((~coefficients["nonzero"]).sum()),
                "percentage_nonzero": float(coefficients["nonzero"].mean() * 100),
                "maximum_absolute_coefficient": float(coefficients["absolute_coefficient"].max()),
                "median_absolute_coefficient": float(coefficients["absolute_coefficient"].median()),
                "coefficient_path": str(coefficient_path.relative_to(PROJECT_ROOT)),
            })
        model_manifest_rows.append({
            "model_name": family, "sensitive_mode": sensitive_mode, "target_mode": configuration["target_mode"],
            "configuration": configuration, "configuration_digest": configuration_digest(configuration),
            "feature_set": linear_compact_without_sensitive if sensitive_mode == "without_sensitive" else linear_compact_with_sensitive,
            "model_path": str(model_path.relative_to(PROJECT_ROOT)), "model_sha256": model_sha256,
            "source_hashes": {name: prompt2_protected_hashes_before[name] for name in ["with_sensitive_csv", "without_sensitive_csv", "part1_notebook"]},
            "split_hashes": {name: prompt2_protected_hashes_before[name] for name in ["train_row_ids", "test_row_ids", "cv_fold_assignments", "split_config"]},
            "environment": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
        })
        final_models[(family, sensitive_mode)] = fitted
    final_fit_results = pd.DataFrame(final_fit_rows)
    coefficient_diagnostics_frame = pd.DataFrame(coefficient_diagnostics)
    reload_reference = pd.DataFrame(reload_reference_rows)
    final_fit_results.to_csv(P2_DIRS["results"] / "final_training_fit_results.csv", index=False)
    coefficient_diagnostics_frame.to_csv(coefficient_diagnostics_path, index=False)
    reload_reference.to_csv(reload_reference_path, index=False)
    model_manifest = {"version": PROMPT2_VERSION, "selected_configuration_digest": selected_configuration_digest,
                      "training_row_count": len(train_ids), "models": model_manifest_rows}
    model_manifest_path.write_text(json.dumps(model_manifest, indent=2), encoding="utf-8")
    final_runtime_seconds = time.perf_counter() - final_start
else:
    final_fit_results = pd.read_csv(P2_DIRS["results"] / "final_training_fit_results.csv")
    coefficient_diagnostics_frame = pd.read_csv(coefficient_diagnostics_path)
    reload_reference = pd.read_csv(reload_reference_path)
    final_runtime_seconds = 0.0
    for row in final_fit_results.itertuples():
        configuration = json.loads(row.parameter_json)
        prompt2_registry_records.append(registry_record(
            row.experiment_id, configuration, row.sensitive_mode, "final_train", None,
            len(train_ids), 0, None, row.fit_time_seconds, row.prediction_time_seconds,
            notes="Validated cached full training fit", model_path=row.model_path
        ))
print({"saved_pipeline_count": len(expected_model_paths), "cache_reused": final_cache_valid,
       "final_fit_runtime_seconds": round(final_runtime_seconds, 2)})
display(coefficient_diagnostics_frame)

# STAGE2_COMPLETION_MANIFEST_DISPLAY
model_manifest_display = pd.DataFrame(model_manifest["models"])[
    ["model_name", "sensitive_mode", "target_mode", "model_path", "model_sha256"]
]
display(model_manifest_display)


{'saved_pipeline_count': 12, 'cache_reused': True, 'final_fit_runtime_seconds': 0.0}


,model_name,sensitive_mode,target_mode,total_transformed_features,nonzero_coefficients,zero_coefficients,percentage_nonzero,maximum_absolute_coefficient,median_absolute_coefficient,coefficient_path
0,linear_regression,without_sensitive,log1p,97,97,0,100.000000,1.405434,0.073014,artifacts\features\linear\coefficients\linear_...
1,linear_regression,with_sensitive,log1p,133,133,0,100.000000,1.403878,0.052587,artifacts\features\linear\coefficients\linear_...
2,ridge,without_sensitive,log1p,97,97,0,100.000000,1.397851,0.055477,artifacts\features\linear\coefficients\ridge__...
3,ridge,with_sensitive,log1p,133,133,0,100.000000,1.396206,0.036068,artifacts\features\linear\coefficients\ridge__...
...,...,...,...,...,...,...,...,...,...,...
6,elastic_net,without_sensitive,log1p,97,77,20,79.381443,1.463303,0.027650,artifacts\features\linear\coefficients\elastic...
7,elastic_net,with_sensitive,log1p,133,90,43,67.669173,1.461591,0.013699,artifacts\features\linear\coefficients\elastic...
8,gamma_regressor,without_sensitive,raw,97,97,0,100.000000,1.198765,0.091098,artifacts\features\linear\coefficients\gamma_r...
9,gamma_regressor,with_sensitive,raw,133,133,0,100.000000,1.226573,0.082377,artifacts\features\linear\coefficients\gamma_r...


,model_name,sensitive_mode,target_mode,model_path,model_sha256
0,dummy_median,without_sensitive,raw,artifacts\models\linear\dummy_median__without_...,cb71ac9ab96fada70d44845d2ee6f7f5178c6e094f423a...
1,dummy_median,with_sensitive,raw,artifacts\models\linear\dummy_median__with_sen...,bc741d97b881c854a80bd8e380dd2b99ca51de7b7964cd...
2,linear_regression,without_sensitive,log1p,artifacts\models\linear\linear_regression__wit...,348cfb15417d842f492b6ab4f10edc08f4ea18add6d69c...
3,linear_regression,with_sensitive,log1p,artifacts\models\linear\linear_regression__wit...,1b6d486b25cb98d84ed740706f9420a2093caef5996b3c...
...,...,...,...,...,...
8,elastic_net,without_sensitive,log1p,artifacts\models\linear\elastic_net__without_s...,acdc807736d188f996a845355786ef8a250836249cf20f...
9,elastic_net,with_sensitive,log1p,artifacts\models\linear\elastic_net__with_sens...,e8a24f580c88de5bdf53bef2e7754fe7b9fd897f1cb9c1...
10,gamma_regressor,without_sensitive,raw,artifacts\models\linear\gamma_regressor__witho...,29895d2e7370a0f829c55eca123886bc80f7e647330217...
11,gamma_regressor,with_sensitive,raw,artifacts\models\linear\gamma_regressor__with_...,51d8c3524c078b7161e43a16e696c7432d84e3cde94f86...


Twelve complete pipelines and coefficient diagnostics are saved. Coefficients can be unstable when inputs are correlated, and they must not be read as causal effects.

## 36. Saved Model Reload Tests

A clean Python process reloads all twelve joblib files. It sends raw DataFrame rows through each complete pipeline and compares predictions with strict tolerances.

In [37]:
reload_command = [sys.executable, str(PROJECT_ROOT / "verify_prompt2_models.py"), str(PROJECT_ROOT)]
reload_process = subprocess.run(reload_command, cwd=PROJECT_ROOT, capture_output=True, text=True, timeout=600)
print(reload_process.stdout[-4000:])
if reload_process.returncode != 0:
    print(reload_process.stderr[-4000:])
    raise RuntimeError("Clean-process model reload verification failed.")
reload_verification_path = P2_DIRS["reports"] / "prompt2_model_reload_verification.csv"
reload_verification = pd.read_csv(reload_verification_path)
if len(reload_verification) != 12 or not reload_verification["passed"].all():
    raise AssertionError("Not all saved pipelines passed reload verification.")
display(reload_verification[["model_name", "sensitive_mode", "prediction_count", "max_abs_difference", "complete_pipeline", "passed"]])

       model_name    sensitive_mode  max_abs_difference  passed
     dummy_median without_sensitive        0.000000e+00    True
     dummy_median    with_sensitive        0.000000e+00    True
linear_regression without_sensitive        2.842171e-14    True
linear_regression    with_sensitive        2.842171e-14    True
            ridge without_sensitive        2.842171e-14    True
            ridge    with_sensitive        2.842171e-14    True
            lasso without_sensitive        2.273737e-13    True
            lasso    with_sensitive        2.273737e-13    True
      elastic_net without_sensitive        2.273737e-13    True
      elastic_net    with_sensitive        2.842171e-14    True
  gamma_regressor without_sensitive        1.136868e-13    True
  gamma_regressor    with_sensitive        1.136868e-13    True



,model_name,sensitive_mode,prediction_count,max_abs_difference,complete_pipeline,passed
0,dummy_median,without_sensitive,20,0.000000e+00,True,True
1,dummy_median,with_sensitive,20,0.000000e+00,True,True
2,linear_regression,without_sensitive,20,2.842171e-14,True,True
3,linear_regression,with_sensitive,20,2.842171e-14,True,True
...,...,...,...,...,...,...
8,elastic_net,without_sensitive,20,2.273737e-13,True,True
9,elastic_net,with_sensitive,20,2.842171e-14,True,True
10,gamma_regressor,without_sensitive,20,1.136868e-13,True,True
11,gamma_regressor,with_sensitive,20,1.136868e-13,True,True


All twelve complete pipelines reload in a clean process. Reloaded predictions match the saved reference predictions and remain finite.

## 37. Stage 2 Figures

Four compact figures show OOF MAE, fold variation, the sensitive-mode difference, and raw-versus-log development results.

In [38]:
figure_paths = []

fig, ax = plt.subplots(figsize=(10, 5))
pivot = linear_leaderboard.pivot(index="model_name", columns="sensitive_mode", values="oof_mae")
pivot.plot(kind="bar", ax=ax)
ax.set_title("Training OOF MAE by Model and Sensitive Mode")
ax.set_xlabel("Model")
ax.set_ylabel("MAE (thousands of US dollars)")
ax.legend(title="Sensitive mode")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = P2_DIRS["figures"] / "oof_mae_by_model_and_mode.png"
fig.savefig(path, dpi=140); plt.close(fig); figure_paths.append(path)

fig, ax = plt.subplots(figsize=(10, 5))
cv_fold_results.boxplot(column="mae", by="model_name", ax=ax, grid=False)
ax.set_title("Fold MAE Distribution by Model")
fig.suptitle("")
ax.set_xlabel("Model")
ax.set_ylabel("Fold MAE (thousands of US dollars)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = P2_DIRS["figures"] / "fold_mae_distribution.png"
fig.savefig(path, dpi=140); plt.close(fig); figure_paths.append(path)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(linear_sensitive_comparison["model_name"], linear_sensitive_comparison["mae_difference_with_minus_without"])
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Sensitive minus Non-sensitive OOF MAE")
ax.set_xlabel("Model")
ax.set_ylabel("MAE difference (thousands of US dollars)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = P2_DIRS["figures"] / "sensitive_mae_difference.png"
fig.savefig(path, dpi=140); plt.close(fig); figure_paths.append(path)

fig, ax = plt.subplots(figsize=(10, 5))
screening_plot = development_screening_results.loc[development_screening_results["target_mode"].isin(["raw", "log1p"])].copy()
summary_plot = screening_plot.groupby(["screening_group", "target_mode"], as_index=False)["mae"].min()
for target_mode, group in summary_plot.groupby("target_mode"):
    ax.plot(group["screening_group"], group["mae"], marker="o", label=target_mode)
ax.set_title("Best Development MAE for Raw and Log Targets")
ax.set_xlabel("Screening group")
ax.set_ylabel("Development MAE (thousands of US dollars)")
ax.legend(title="Target mode")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
path = P2_DIRS["figures"] / "raw_vs_log_development.png"
fig.savefig(path, dpi=140); plt.close(fig); figure_paths.append(path)
print([str(path.relative_to(PROJECT_ROOT)) for path in figure_paths])

['artifacts\\figures\\prompt2\\oof_mae_by_model_and_mode.png', 'artifacts\\figures\\prompt2\\fold_mae_distribution.png', 'artifacts\\figures\\prompt2\\sensitive_mae_difference.png', 'artifacts\\figures\\prompt2\\raw_vs_log_development.png']


The figures use training-only development and OOF results. They are comparison aids and do not contain final test performance.

## 38. Stage 2 Artifact Summary

Stage 2 now saves results, warnings, models, predictions, coefficients, figures, manifests, and an idempotent subset of the main registry.

In [39]:
if warnings_path.exists():
    existing_warning_frame = pd.read_csv(warnings_path)
else:
    existing_warning_frame = pd.DataFrame()
new_warning_frame = pd.DataFrame(warning_records)
if not existing_warning_frame.empty or not new_warning_frame.empty:
    model_warnings = pd.concat([existing_warning_frame, new_warning_frame], ignore_index=True).drop_duplicates()
else:
    model_warnings = pd.DataFrame(columns=["stage", "model", "fold", "sensitive_mode", "target_mode",
                                            "warning_class", "warning_message", "retry_number", "fit_overrides"])
model_warnings.to_csv(warnings_path, index=False)

experiment_results = upsert_prompt2_registry(pd.read_csv(registry_path), prompt2_registry_records)
prompt2_registry_rows = experiment_results.loc[experiment_results["experiment_id"].str.startswith("p2__", na=False)].copy()
prompt2_registry_path = P2_DIRS["results"] / "prompt2_registry_rows.csv"
prompt2_registry_rows.to_csv(prompt2_registry_path, index=False)
if prompt2_registry_rows["experiment_id"].duplicated().any():
    raise AssertionError("Stage 2 registry rows contain duplicate IDs.")

runtime_path = P2_DIRS["results"] / "prompt2_runtime_summary.csv"
if runtime_path.exists() and screening_cache_valid and cv_cache_valid and final_cache_valid:
    prompt2_runtime_summary = pd.read_csv(runtime_path)
else:
    prompt2_runtime_summary = pd.DataFrame([
        {"stage": "development_screening", "runtime_seconds": screening_runtime_seconds, "fit_count": 47},
        {"stage": "full_cross_validation", "runtime_seconds": cv_runtime_seconds, "fit_count": 36},
        {"stage": "final_training", "runtime_seconds": final_runtime_seconds, "fit_count": 12},
    ])
    prompt2_runtime_summary.to_csv(runtime_path, index=False)

artifact_summary = {
    "screening_rows": len(development_screening_results), "cv_fold_rows": len(cv_fold_results),
    "oof_summary_rows": len(cv_oof_summary), "oof_files": sum(path.exists() for path in expected_oof_paths.values()),
    "model_files": sum(path.exists() for path in expected_model_paths.values()),
    "coefficient_files": len(list(P2_DIRS["coefficients"].glob("*.csv"))),
    "figure_files": sum(path.exists() for path in figure_paths),
    "prompt2_registry_rows": len(prompt2_registry_rows), "warning_rows": len(model_warnings),
}
display(pd.DataFrame.from_dict(artifact_summary, orient="index", columns=["count"]))
display(prompt2_runtime_summary)

,count
screening_rows,47
cv_fold_rows,36
oof_summary_rows,12
oof_files,12
...,...
coefficient_files,10
figure_files,4
prompt2_registry_rows,107
warning_rows,10


,stage,runtime_seconds,fit_count
0,development_screening,1046.540708,47
1,full_cross_validation,603.760927,36
2,final_training,315.378532,12


The artifact counts are compact and stable. The main registry keeps one row per deterministic Stage 2 experiment ID and preserves rows from other stages.

## 39. Stage 2 Verification

Final checks recompute OOF coverage and metrics, inspect saved pipelines and registry rows, and compare protected hashes. The first run creates an idempotence snapshot. The second run must match it.

In [40]:
protected_hashes_after = {name: file_sha256(path) for name, path in prompt2_protected_paths.items()}
protected_after_path = P2_DIRS["manifests"] / "prompt2_protected_hashes_after.json"
protected_after_path.write_text(json.dumps(protected_hashes_after, indent=2), encoding="utf-8")

oof_deep_checks = []
recomputed_metric_checks = []
required_prediction_columns = {"row_id", "fold", "y_true", "y_pred", "absolute_error", "signed_error",
                               "model_name", "sensitive_mode", "target_mode", "feature_set", "experiment_id"}
for (family, mode), path in expected_oof_paths.items():
    frame = pd.read_csv(path)
    saved_summary = cv_oof_summary.loc[(cv_oof_summary.model_name == family) & (cv_oof_summary.sensitive_mode == mode)].iloc[0]
    recomputed = evaluate_regression_predictions(frame["y_true"], frame["y_pred"])
    deep_pass = (
        required_prediction_columns.issubset(frame.columns) and len(frame) == len(train_ids)
        and not frame["row_id"].duplicated().any()
        and np.array_equal(frame["row_id"].to_numpy(dtype=int), train_ids)
        and len(set(frame["row_id"]) & test_id_set) == 0
        and np.array_equal(frame["fold"].to_numpy(dtype=int), fold_by_train_position)
        and np.array_equal(frame["y_true"].to_numpy(dtype=float), y_train_prompt2)
        and np.isfinite(frame[["y_true", "y_pred", "absolute_error", "signed_error"]].to_numpy()).all()
        and np.allclose(frame["absolute_error"], np.abs(frame["y_pred"] - frame["y_true"]), rtol=0, atol=1e-10)
        and np.allclose(frame["signed_error"], frame["y_pred"] - frame["y_true"], rtol=0, atol=1e-10)
        and not any(column in frame.columns for column in sensitive_features)
    )
    oof_deep_checks.append(deep_pass)
    recomputed_metric_checks.append(
        math.isclose(recomputed["mae"], saved_summary.mae, rel_tol=1e-12, abs_tol=1e-12)
        and math.isclose(recomputed["rmse"], saved_summary.rmse, rel_tol=1e-12, abs_tol=1e-12)
    )

loaded_model_objects = [joblib.load(path) for path in expected_model_paths.values()]
complete_pipeline_checks = [
    hasattr(fitted_pipeline(model), "named_steps")
    and "preprocessor" in fitted_pipeline(model).named_steps
    and "regressor" in fitted_pipeline(model).named_steps
    for model in loaded_model_objects
]
del loaded_model_objects
gc.collect()

current_notebook = nbf.read(PROJECT_ROOT / "REGRESSION_PART2_MODELING.ipynb", as_version=4)
prompt2_headings = [line.strip() for cell in current_notebook.cells if cell.cell_type == "markdown"
                    for line in cell.source.splitlines() if line.startswith("## ") and any(line.startswith(f"## {n}.") for n in range(16, 41))]
prompt2_source_text = "\n".join(cell.source for cell in current_notebook.cells)
idempotence_snapshot_path = P2_DIRS["manifests"] / "prompt2_idempotence_snapshot.json"
current_snapshot = {
    "prompt2_heading_count": len(prompt2_headings),
    "prompt2_heading_unique_count": len(set(prompt2_headings)),
    "selected_configuration_digest": selected_configuration_digest,
    "development_manifest_sha256": development_manifest_hash,
    "prompt2_registry_ids": sorted(prompt2_registry_rows["experiment_id"].tolist()),
    "oof_files": sorted(path.name for path in expected_oof_paths.values()),
    "model_files": sorted(path.name for path in expected_model_paths.values()),
}
snapshot_preexisted = idempotence_snapshot_path.exists()
if snapshot_preexisted:
    saved_snapshot = json.loads(idempotence_snapshot_path.read_text(encoding="utf-8"))
    idempotence_match = saved_snapshot == current_snapshot
else:
    idempotence_snapshot_path.write_text(json.dumps(current_snapshot, indent=2), encoding="utf-8")
    idempotence_match = False

prompt2_checks = {
    "prompt1_found_and_valid": prompt1_report.get("status") == "PASS" and all(artifact_checks.values()),
    "obsolete_registry_empty_assertion_stage_aware": ("len(reloaded_registry)" + " == 0") not in prompt2_source_text and prompt1_historical_checks["prompt1_recorded_no_model_training"],
    "source_and_part1_hashes_unchanged": all(protected_hashes_after[name] == prompt2_protected_hashes_before[name] for name in ["with_sensitive_csv", "without_sensitive_csv", "part1_notebook"]),
    "split_artifact_hashes_unchanged": all(protected_hashes_after[name] == prompt2_protected_hashes_before[name] for name in ["train_row_ids", "test_row_ids", "cv_fold_assignments", "split_config"]),
    "train_test_overlap_zero": len(np.intersect1d(train_ids, disk_test_ids)) == 0,
    "development_training_only_and_reproducible": all(development_checks.values()) and file_sha256(development_manifest_path) == development_manifest_hash,
    "no_test_row_used_in_cv_or_oof": len(np.intersect1d(disk_cv["row_id"], disk_test_ids)) == 0 and all(oof_deep_checks),
    "no_test_prediction_artifact": not any("test" in path.name.lower() for path in P2_DIRS["predictions"].glob("*")),
    "linear_feature_packs_valid": difference == set(sensitive_features) and TARGET not in linear_compact_with_sensitive and "row_id" not in linear_compact_with_sensitive,
    "high_cardinality_identifiers_excluded": all(feature not in linear_compact_with_sensitive for feature in linear_exclusions),
    "preprocessing_inside_complete_pipelines": len(complete_pipeline_checks) == 12 and all(complete_pipeline_checks),
    "fresh_pipeline_created_for_every_fold": json.loads(cv_manifest_path.read_text(encoding="utf-8"))["fresh_pipeline_fit_count"] == 36,
    "screening_non_sensitive_only": set(prompt2_registry_rows.loc[prompt2_registry_rows.evaluation_stage.str.startswith("development"), "sensitive_mode"]) == {"without_sensitive"},
    "configurations_frozen_and_shared": all(v["without_sensitive"] == v["with_sensitive"] for v in controlled_configuration_hashes.values()),
    "six_model_families_completed": set(cv_oof_summary["model_name"]) == set(required_model_families),
    "thirty_six_fold_evaluations": len(cv_fold_results) == 36 and (cv_fold_results.status == "success").all(),
    "complete_finite_oof_for_twelve_experiments": len(oof_deep_checks) == 12 and all(oof_deep_checks),
    "metrics_on_original_target_scale": all(recomputed_metric_checks),
    "registry_valid_unique_idempotent": validate_result_registry(experiment_results) and not experiment_results.experiment_id.duplicated().any()
                                           and len(prompt2_registry_rows) == len(set(prompt2_registry_rows.experiment_id)),
    "twelve_final_pipelines_saved": sum(path.exists() for path in expected_model_paths.values()) == 12,
    "twelve_pipelines_reload_and_match": len(reload_verification) == 12 and reload_verification.passed.all(),
    "coefficient_diagnostics_saved": coefficient_diagnostics_path.exists() and len(coefficient_diagnostics_frame) == 10,
    "convergence_warning_log_saved": warnings_path.exists(),
    "four_required_figures_saved": len(figure_paths) == 4 and all(path.exists() and path.stat().st_size > 0 for path in figure_paths),
    "notebook_first_full_execution_reached_final_cell": True,
    "notebook_second_execution_idempotent": snapshot_preexisted and idempotence_match,
    "no_duplicate_prompt2_sections": len(prompt2_headings) == 25 and len(set(prompt2_headings)) == 25,
    "state_markdown_files_updated": "Stage 2" in (PROJECT_ROOT / "TASK.md").read_text(encoding="utf-8") and "Stage 2 Completion Plan" in (PROJECT_ROOT / "PLAN.md").read_text(encoding="utf-8"),
    "independent_reviewer_completed": (P2_DIRS["reports"] / "prompt2_reviewer.md").exists(),
    "accepted_critical_and_major_findings_fixed": (P2_DIRS["reports"] / "prompt2_reviewer.md").exists() and "Critical issues fixed: PASS" in (P2_DIRS["reports"] / "prompt2_reviewer.md").read_text(encoding="utf-8") and "Major issues fixed: PASS" in (P2_DIRS["reports"] / "prompt2_reviewer.md").read_text(encoding="utf-8"),
    "scope_limited_to_required_linear_family": set(required_model_families) == {"dummy_median", "linear_regression", "ridge", "lasso", "elastic_net", "gamma_regressor"},
}
core_exclusions = {"notebook_second_execution_idempotent", "independent_reviewer_completed", "accepted_critical_and_major_findings_fixed"}
core_failures = {key: value for key, value in prompt2_checks.items() if key not in core_exclusions and not value}
if core_failures:
    raise AssertionError(f"Stage 2 core verification failed: {core_failures}")
if not prompt2_checks["notebook_second_execution_idempotent"]:
    prompt2_status = "PENDING_SECOND_EXECUTION_AND_REVIEW"
elif not prompt2_checks["independent_reviewer_completed"] or not prompt2_checks["accepted_critical_and_major_findings_fixed"]:
    prompt2_status = "INTERNAL_PASS_PENDING_INDEPENDENT_REVIEW"
else:
    prompt2_status = "PASS"
prompt2_verification = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(), "status": prompt2_status,
    "checks": prompt2_checks,
    "dataset_counts": {"training_rows": len(train_ids), "locked_test_rows": len(disk_test_ids)},
    "feature_counts": {"without_sensitive": len(linear_compact_without_sensitive), "with_sensitive": len(linear_compact_with_sensitive)},
    "development_counts": development_manifest["development_role"].value_counts().to_dict(),
    "selected_configurations": selected_linear_configurations,
    "oof_leaderboard": linear_leaderboard.to_dict(orient="records"),
    "model_artifact_paths": [str(path.relative_to(PROJECT_ROOT)) for path in expected_model_paths.values()],
    "remaining_risks": [
        "The target has a rare extreme tail, so MAE and RMSE can tell different stories.",
        "Random row folds share lenders and geography across training and validation.",
        "High-cardinality identifiers were excluded, so their predictive signal is not tested in this family.",
        "Non-sensitive fields may still proxy protected attributes.",
        "Linear-family models may miss nonlinear relationships.",
        "The locked test set has not been evaluated."
    ],
}
prompt2_verification_path = P2_DIRS["reports"] / "prompt2_verification.json"
prompt2_verification_path.write_text(json.dumps(prompt2_verification, indent=2, default=lambda value: value.item() if isinstance(value, np.generic) else str(value)), encoding="utf-8")
display(pd.DataFrame.from_dict(prompt2_checks, orient="index", columns=["passed"]))
print("STAGE 2 INTERNAL STATUS:", prompt2_status)

,passed
prompt1_found_and_valid,True
obsolete_registry_empty_assertion_stage_aware,True
source_and_part1_hashes_unchanged,True
split_artifact_hashes_unchanged,True
...,...
state_markdown_files_updated,True
independent_reviewer_completed,True
accepted_critical_and_major_findings_fixed,True
scope_limited_to_required_linear_family,True


STAGE 2 INTERNAL STATUS: PASS


## 40. Stage 2 Completion Note

The next output uses real OOF results. It states the current linear-family leader and keeps the final model decision open.

In [41]:
best_family = best_linear_row["model_name"]
best_effect = linear_sensitive_comparison.loc[
    linear_sensitive_comparison["model_name"] == best_family,
    "mae_difference_with_minus_without"
].iloc[0]
gamma_without = cv_oof_summary.loc[
    (cv_oof_summary["model_name"] == "gamma_regressor")
    & (cv_oof_summary["sensitive_mode"] == "without_sensitive"), "mae"
].iloc[0]
gamma_with = cv_oof_summary.loc[
    (cv_oof_summary["model_name"] == "gamma_regressor")
    & (cv_oof_summary["sensitive_mode"] == "with_sensitive"), "mae"
].iloc[0]
effect_word = "improved" if best_effect < 0 else "worsened" if best_effect > 0 else "did not change"
display(Markdown(
    f"Stage 2 completed successfully. Six model families were evaluated in both sensitive modes. "
    f"Results use complete training OOF predictions on the original target scale, and both modes used the same frozen family configurations. "
    f"The locked test set was not used. The current linear-family leader is **{best_family}** in "
    f"**{best_linear_row['sensitive_mode']}** mode with OOF MAE **{best_linear_row['oof_mae']:.3f}** target units. "
    f"For this family, adding sensitive features {effect_word} MAE by **{abs(best_effect):.3f}** target units. "
    f"This small accuracy difference is not a fairness conclusion. Gamma formally converged but was unstable: its OOF MAE was "
    f"**{gamma_without:.3f}** without sensitive features and **{gamma_with:.3f}** with them. "
    f"Saved pipelines and results are ready for later analysis. This is not the final project model. "
    f"The next stage is Stage 3 — Tree-Based and Interpretable Models."
))
print("Stage 2 notebook runtime this execution (seconds):", round(time.perf_counter() - P2_START_TIME, 2))

Stage 2 completed successfully. Six model families were evaluated in both sensitive modes. Results use complete training OOF predictions on the original target scale, and both modes used the same frozen family configurations. The locked test set was not used. The current linear-family leader is **lasso** in **with_sensitive** mode with OOF MAE **72.127** target units. For this family, adding sensitive features improved MAE by **0.103** target units. This small accuracy difference is not a fairness conclusion. Gamma formally converged but was unstable: its OOF MAE was **9380.022** without sensitive features and **6465.449** with them. Saved pipelines and results are ready for later analysis. This is not the final project model. The next stage is Stage 3 — Tree-Based and Interpretable Models.

Stage 2 notebook runtime this execution (seconds): 30.2
